# Day 3 — LangChain Core

## Part 1 — Why LangChain Exists and the Problems It Solves

---

# Before We Learn LangChain

Most beginners make a mistake when learning AI engineering.

They see LangChain and immediately start learning:

- Chains
- Runnables
- Agents
- Retrieval
- LCEL

without understanding:

> Why did LangChain need to exist in the first place?

If you don't understand the problem, every LangChain abstraction looks arbitrary.

If you understand the problem, every abstraction feels inevitable.

So before touching a single LangChain component, we must travel back in time.

---

# The World Before LangChain

Imagine it is early 2023.

LLMs have become powerful.

People are building applications using:

- OpenAI APIs
- Anthropic APIs
- Local Models
- HuggingFace Models

The model itself is amazing.

The surrounding infrastructure is not.

Every developer is writing the same boilerplate code repeatedly.

A simple question-answering application might require:

```python
# Build prompt

prompt = f"""
Answer the question.

Question:
{question}
"""

# Call model

response = client.chat.completions.create(
    model="gpt-4",
    messages=[
        {"role":"user","content":prompt}
    ]
)

# Extract output

answer = response.choices[0].message.content

return answer
```

Seems fine.

Now let's add memory.

---

# Adding Memory

Suddenly the application becomes:

```python
history.append(user_message)

prompt = f"""
Conversation:

{history}

User:
{user_message}
"""

response = llm(prompt)

history.append(response)

return response
```

Still manageable.

Now let's add tools.

---

# Adding Tool Calling

Suppose the user asks:

```text
What's the weather in Lahore?
```

The model cannot access weather data.

So we build:

```python
if "weather" in query:
    result = weather_tool(query)

    prompt = f"""
    Tool Result:

    {result}

    Answer the user.
    """

    answer = llm(prompt)
```

Now things become more complicated.

---

# Adding Multiple Tools

Now we add:

- Calculator
- Search
- Weather
- Database Query

Suddenly we need routing.

```python
if weather:
    weather_tool()

elif calculator:
    calculator_tool()

elif database:
    database_tool()
```

The codebase starts growing.

---

# Adding Retrieval

Now we want the AI to answer questions from documents.

We need:

1. Load document
2. Split document
3. Generate embeddings
4. Store vectors
5. Search vectors
6. Inject context
7. Generate answer

Suddenly the flow becomes:

```text
User Question
      │
      ▼
Embedding
      │
      ▼
Vector Search
      │
      ▼
Retrieve Chunks
      │
      ▼
Prompt Construction
      │
      ▼
LLM
      │
      ▼
Answer
```

This is no longer a simple API call.

It is now a system.

---

# The Real Problem

The LLM itself was never the hard part.

The hard part was orchestration.

---

## What Is Orchestration?

Orchestration means:

> Coordinating many AI-related components into a single workflow.

Example:

```text
User Query
     │
     ▼
Prompt
     │
     ▼
LLM
     │
     ▼
Tool
     │
     ▼
Database
     │
     ▼
Memory
     │
     ▼
Response
```

Each component has dependencies.

Each component has inputs and outputs.

Managing these manually becomes painful.

---

# The Software Engineering Problem

Imagine building ten AI applications.

Without a framework, every project contains:

```python
prompt building
model calls
tool calls
memory management
vector search
routing logic
error handling
```

repeated over and over.

This violates one of the most important software engineering principles:

> Don't Repeat Yourself (DRY)

Developers were reinventing the wheel.

Again.

And again.

And again.

---

# The Need For Abstractions

Software engineering has a recurring pattern.

Whenever developers repeatedly solve the same problem:

an abstraction emerges.

Examples:

| Problem | Abstraction |
|----------|------------|
| Raw Machine Code | Programming Languages |
| SQL Connections | ORMs |
| HTTP Requests | Web Frameworks |
| Neural Networks | PyTorch |
| AI Workflows | LangChain |

LangChain is simply another abstraction layer.

---

# First-Principles Definition Of LangChain

LangChain is:

> A framework for composing LLM-powered applications using reusable building blocks.

Notice something important.

LangChain is NOT:

- an LLM
- a model provider
- a vector database
- an embedding model

Instead:

LangChain sits between all of them.

```text
Application
     │
     ▼
 LangChain
 ┌───────────┐
 │ Prompts   │
 │ Chains    │
 │ Agents    │
 │ Memory    │
 │ Retrieval │
 └───────────┘
     │
     ▼
LLM Providers
(OpenAI/Groq/etc)
```

---

# Why Chains Were Invented

Now we reach the first major idea.

Imagine this code:

```python
result1 = step1()

result2 = step2(result1)

result3 = step3(result2)

result4 = step4(result3)
```

This pattern appears everywhere.

Output of one component becomes input of another.

This is called a pipeline.

LangChain introduced the concept of a Chain.

A chain is simply:

```text
Output A
    │
    ▼
Input B
    │
    ▼
Output B
    │
    ▼
Input C
```

Instead of manually wiring everything together every time.

---

# Real-World Analogy: Assembly Line

Imagine a car factory.

Station 1:

```text
Build Frame
```

Station 2:

```text
Install Engine
```

Station 3:

```text
Paint Vehicle
```

Station 4:

```text
Quality Check
```

Each station receives the output from the previous station.

This is exactly what a chain does.

```text
Prompt
   ▼
LLM
   ▼
Parser
   ▼
Output
```

---

# Why Chains Became Popular

Chains solved three problems.

## Problem 1 — Readability

Without chains:

```python
x = f1()

y = f2(x)

z = f3(y)

a = f4(z)
```

With chains:

```text
f1 → f2 → f3 → f4
```

Much easier to understand.

---

## Problem 2 — Reusability

You can reuse components.

```python
prompt
llm
parser
```

across many applications.

---

## Problem 3 — Modularity

Replace one component without changing everything else.

Example:

```text
OpenAI
```

becomes

```text
Groq
```

while the rest of the workflow remains unchanged.

---

# The Problem With Early LangChain

Early LangChain worked.

But developers noticed something.

Large workflows became difficult to manage.

Example:

```python
chain = (
    prompt
    -> model
    -> parser
)
```

was fine.

But complex workflows became messy.

Developers wanted:

- branching
- parallelism
- transformations
- routing

without writing large amounts of custom code.

This led to LCEL.

---

# What Is LCEL?

LCEL stands for:

```text
LangChain Expression Language
```

Think of it as:

> A language for describing AI workflows.

---

# The Core Idea Behind LCEL

Instead of writing:

```python
step1()
step2()
step3()
step4()
```

you declare relationships between components.

Example:

```text
Prompt
   │
   ▼
Model
   │
   ▼
Parser
```

LCEL turns workflows into composable objects.

---

# What Manual Code Does LCEL Replace?

Without LCEL:

```python
prompt = create_prompt(question)

response = llm.invoke(prompt)

answer = parser(response)

return answer
```

With LCEL:

```python
chain = prompt | llm | parser
```

The logic is identical.

The abstraction is cleaner.

---

# Why LCEL Was Revolutionary

Before LCEL:

Developers described execution.

```python
do this
then this
then this
```

After LCEL:

Developers described flow.

```text
Prompt
   │
   ▼
LLM
   │
   ▼
Parser
```

This is closer to how humans think about systems.

---

# The Bigger Vision

LangChain was gradually evolving toward a new idea.

Instead of writing AI systems as procedural code:

```python
step1()
step2()
step3()
```

developers could describe systems as graphs.

```text
Node A
  │
  ▼
Node B
  │
  ▼
Node C
```

This idea later became the foundation for modern agent frameworks such as:

- LangGraph
- CrewAI
- AutoGen
- Multi-Agent Systems

---

# Tradeoffs Of Using A Framework

Every abstraction has a cost.

---

## Advantages

### Faster Development

Common patterns already exist.

### Cleaner Code

Less boilerplate.

### Easier Maintenance

Reusable components.

### Better Collaboration

Teams share conventions.

### Rapid Prototyping

Build applications quickly.

---

## Disadvantages

### Hidden Complexity

Frameworks hide implementation details.

Beginners often don't understand what is happening underneath.

---

### Debugging Can Be Harder

Manual code:

```python
print(x)
```

Framework code:

```python
chain.invoke(...)
```

may hide internal execution.

---

### Framework Lock-In

If the framework changes:

your code may need updates.

---

### Learning Curve

You must learn:

- LangChain concepts
- LangChain APIs
- LangChain conventions

before becoming productive.

---

# How LangChain Relates To Day 2's Manual ReAct Agent

This is the most important section.

Many students mistakenly think:

```text
Manual Agent
```

and

```text
LangChain Agent
```

are fundamentally different.

They are not.

The architecture is identical.

---

## Your Manual ReAct Agent

Day 2 looked something like:

```text
User Query
      │
      ▼
Reasoning
      │
      ▼
Tool Selection
      │
      ▼
Tool Execution
      │
      ▼
Observation
      │
      ▼
Reasoning
      │
      ▼
Final Answer
```

---

## LangChain Agent

LangChain does:

```text
User Query
      │
      ▼
Reasoning
      │
      ▼
Tool Selection
      │
      ▼
Tool Execution
      │
      ▼
Observation
      │
      ▼
Reasoning
      │
      ▼
Final Answer
```

Exactly the same architecture.

The difference is:

LangChain provides reusable abstractions.

---

# Mental Model For The Rest Of Day 3

As we move forward, remember:

| Concept | Reality |
|----------|----------|
| Chain | Workflow |
| Runnable | Executable Component |
| LCEL | Workflow Language |
| Prompt Template | Prompt Generator |
| Output Parser | Output Cleaner |
| Retriever | Context Searcher |
| Agent | ReAct Loop Abstraction |

Nothing magical is happening.

LangChain is mostly:

> Packaging patterns that AI engineers were already writing manually.

The goal is not to replace understanding.

The goal is to reduce repetitive engineering work while preserving the same underlying AI concepts you learned in Days 1 and 2.

---

# Interview-Level Questions

### Q1: Why was LangChain created?

Because AI applications require orchestration of prompts, models, tools, memory, retrieval systems, and output processing. Managing these manually becomes repetitive and difficult to scale.

---

### Q2: What problem do Chains solve?

They standardize multi-step workflows where the output of one component becomes the input of another.

---

### Q3: What problem does LCEL solve?

It provides a declarative way to compose AI workflows instead of manually wiring every function call.

---

### Q4: Is LangChain necessary?

No.

Everything LangChain does can be implemented manually in Python.

LangChain primarily improves:

- developer productivity
- readability
- modularity
- maintainability

---

### Q5: What is the relationship between LangChain and ReAct?

LangChain does not replace ReAct.

It provides abstractions that make ReAct-style systems easier to build and maintain.


# Day 3 — LangChain Core

## Part 2 — LangChain Architecture and the Runnable Abstraction

---

# Before Learning Runnables

Most students learn LangChain backwards.

They memorize:

```python
prompt | llm | parser
```

without understanding:

- Why this syntax exists
- What actually executes
- What a Runnable really is
- Why LCEL was able to unify the entire framework

The result is predictable.

They can copy tutorials.

They cannot build systems.

Today we fix that.

---

# The Most Important Idea In Modern LangChain

Everything revolves around a single abstraction:

```text
Runnable
```

If you deeply understand Runnables, then:

- LCEL becomes obvious
- Chains become obvious
- Retrieval becomes obvious
- Agents become obvious

Almost every major LangChain component is built on top of this idea.

---

# Life Before Runnables

Imagine we have three independent functions.

```python
def make_prompt(question):
    return f"Answer:\n{question}"

def call_llm(prompt):
    return llm.invoke(prompt)

def parse_output(response):
    return response.content
```

To execute them:

```python
prompt = make_prompt(question)

response = call_llm(prompt)

answer = parse_output(response)
```

This works.

But there is a problem.

Each component behaves differently.

---

# The Interface Problem

Consider:

```python
PromptTemplate
```

It behaves one way.

```python
ChatModel
```

behaves another way.

```python
Retriever
```

behaves another way.

```python
OutputParser
```

behaves another way.

Each object has:

- different methods
- different inputs
- different outputs

This creates integration headaches.

---

# Software Engineering Principle

When many objects perform similar jobs:

we create a common interface.

Example:

```python
Dog.run()

Cat.run()

Human.run()
```

Different implementations.

Same interface.

---

# The Runnable Idea

LangChain asked:

> What if every component in the framework could be executed in exactly the same way?

The answer became:

```text
Runnable
```

A Runnable is simply:

> Any object that accepts input and produces output.

---

# First-Principles Definition

Mathematically:

```text
Input
  │
  ▼
Runnable
  │
  ▼
Output
```

That's it.

Nothing more.

Nothing less.

---

# Real-World Analogy

Imagine a factory machine.

```text
Raw Material
      │
      ▼
Machine
      │
      ▼
Processed Material
```

The machine does not care what came before.

The machine does not care what comes after.

Its job is:

```text
Input → Transformation → Output
```

A Runnable works exactly the same way.

---

# Why This Was Revolutionary

Before Runnables:

```python
prompt.format()

llm.generate()

retriever.search()

parser.parse()
```

Different APIs.

Different execution methods.

Different conventions.

---

After Runnables:

Everything can be executed with:

```python
.invoke()
```

Example:

```python
prompt.invoke(data)

llm.invoke(prompt)

parser.invoke(output)

retriever.invoke(query)
```

Suddenly the entire framework becomes consistent.

---

# The Universal Contract

Every Runnable follows the same contract.

```text
Receive Input
      │
      ▼
Process Input
      │
      ▼
Return Output
```

This universal contract enables composition.

---

# Why Composition Matters

Suppose:

```python
A(input)
```

returns:

```python
output_a
```

and

```python
B(output_a)
```

accepts that output.

Then:

```text
A → B
```

can be connected automatically.

This is the foundation of LCEL.

---

# Visualizing Runnable Composition

Without composition:

```python
x = A(data)

y = B(x)

z = C(y)
```

With composition:

```text
A
│
▼
B
│
▼
C
```

The workflow itself becomes the object.

---

# The Four Core Runnable Operations

LangChain eventually standardized around four major execution methods.

---

## 1. invoke()

Single Input

Single Output

```python
result = runnable.invoke(data)
```

Example:

```python
answer = llm.invoke("What is AI?")
```

Flow:

```text
Question
    │
    ▼
 Runnable
    │
    ▼
 Answer
```

---

## 2. batch()

Multiple Inputs

Multiple Outputs

Instead of:

```python
for question in questions:
    llm.invoke(question)
```

you can do:

```python
llm.batch(questions)
```

Flow:

```text
Q1
Q2
Q3
Q4
 │
 ▼
Runnable
 │
 ▼
A1
A2
A3
A4
```

---

## 3. stream()

Receive tokens gradually.

Instead of:

```text
(wait)
(wait)
(wait)

Full Response
```

You receive:

```text
H
He
Hel
Hell
Hello
```

token by token.

---

## 4. ainvoke()

Async execution.

Useful when:

- APIs are slow
- Multiple calls happen simultaneously
- High-performance applications

---

# Understanding LCEL From First Principles

Now that we understand Runnables:

LCEL becomes easy.

---

# What Is LCEL Really?

Remember:

A Runnable produces output.

Another Runnable accepts input.

Therefore:

they can be connected.

---

Example:

```text
Prompt Template
        │
        ▼
Chat Model
        │
        ▼
Parser
```

Since every component is a Runnable:

they can form a pipeline.

---

# The Pipe Operator

LCEL uses:

```python
|
```

which resembles Unix pipes.

Example:

```python
chain = prompt | llm | parser
```

Read this as:

```text
Take output from prompt
and feed it to llm

Take output from llm
and feed it to parser
```

---

# Internal Execution

Suppose:

```python
chain = A | B | C
```

User calls:

```python
chain.invoke(x)
```

Internally:

```python
temp1 = A.invoke(x)

temp2 = B.invoke(temp1)

result = C.invoke(temp2)
```

This is all LCEL is doing.

No magic.

Just automated wiring.

---

# Manual Python Equivalent

LCEL:

```python
chain = prompt | llm | parser
```

Manual Version:

```python
formatted_prompt = prompt.invoke(data)

response = llm.invoke(formatted_prompt)

final_answer = parser.invoke(response)
```

Exactly the same.

---

# Why LCEL Became Popular

Because workflows become visually obvious.

Compare:

```python
formatted_prompt = prompt.invoke(data)

response = llm.invoke(formatted_prompt)

answer = parser.invoke(response)
```

versus

```python
prompt | llm | parser
```

The second version immediately communicates architecture.

---

# The LangChain Architecture

Now we can understand the entire framework.

---

## Layer 1 — Inputs

Examples:

```text
User Message
Document
Database Record
API Response
```

---

## Layer 2 — Prompt Layer

Transforms raw data into prompts.

```text
Input
  │
  ▼
PromptTemplate
```

---

## Layer 3 — Model Layer

Calls the LLM.

```text
Prompt
   │
   ▼
Chat Model
```

Examples:

- Groq
- OpenAI
- Anthropic
- Gemini

---

## Layer 4 — Processing Layer

Parses output.

```text
LLM Output
     │
     ▼
Parser
```

---

## Layer 5 — Retrieval Layer

Provides context.

```text
Question
    │
    ▼
Retriever
    │
    ▼
Relevant Chunks
```

---

## Layer 6 — Agent Layer

Adds reasoning.

```text
Reason
  │
  ▼
Tool
  │
  ▼
Observe
```

which should remind you of ReAct.

---

# How This Connects To Day 2

Day 2:

```python
thought()

tool()

observation()

final_answer()
```

---

Day 3:

```python
prompt
|
llm
|
parser
```

Same philosophy.

Different abstraction level.

---

# The Hidden Genius Of Runnables

The Runnable abstraction solved a major engineering challenge.

Instead of creating separate APIs for:

- prompts
- models
- retrievers
- parsers
- chains
- agents

LangChain made everything behave similarly.

This allows:

```python
A | B | C | D | E
```

regardless of what those components actually are.

---

# Common Beginner Mistake #1

Thinking LCEL is a new AI algorithm.

It is not.

LCEL is merely a workflow description language.

---

# Common Beginner Mistake #2

Thinking LangChain performs reasoning.

The LLM performs reasoning.

LangChain performs orchestration.

---

# Common Beginner Mistake #3

Thinking Chains are intelligent.

Chains are deterministic pipelines.

They simply move data between components.

---

# Common Beginner Mistake #4

Confusing Agents and Chains

Chain:

```text
Fixed Workflow
```

Agent:

```text
Dynamic Workflow
```

A chain knows its path beforehand.

An agent decides its path while running.

---

# Interview-Level Understanding

## Question 1

What is a Runnable?

Answer:

A Runnable is a standardized executable component that accepts input and produces output.

---

## Question 2

Why was the Runnable abstraction introduced?

Answer:

To unify execution across prompts, models, parsers, retrievers, chains, and agents through a common interface.

---

## Question 3

What does LCEL replace?

Answer:

Manual wiring of outputs and inputs between workflow components.

---

## Question 4

What happens internally when executing:

```python
A | B | C
```

Answer:

Equivalent to:

```python
result1 = A.invoke(input)

result2 = B.invoke(result1)

result3 = C.invoke(result2)
```

---

## Question 5

Does LCEL make models smarter?

Answer:

No.

LCEL improves workflow composition, not model intelligence.

---

# Mental Model To Carry Forward

Before moving to the next chapter, remember:

```text
Everything Is A Runnable
```

Prompt:

```text
Runnable
```

LLM:

```text
Runnable
```

Parser:

```text
Runnable
```

Retriever:

```text
Runnable
```

Chain:

```text
Runnable
```

Agent:

```text
Runnable
```

Once you understand that one sentence, most of modern LangChain starts making sense.



## Part 3 — Prompt Templates, Chat Models, and Output Parsers

---

# Before We Write Any LangChain Code

At first glance, these three components seem unrelated.

```text
Prompt Template
Chat Model
Output Parser
```

Most tutorials teach them separately.

That is a mistake.

In reality they form a pipeline.

```text
Raw Input
     │
     ▼
Prompt Template
     │
     ▼
Chat Model
     │
     ▼
Output Parser
     │
     ▼
Final Structured Result
```

Before learning the implementation, we must understand:

> Why do these components exist at all?

---

# The Problem Prompt Templates Solve

Suppose we are building an AI tutor.

A user asks:

```text
What is a Transformer?
```

We could write:

```python
prompt = f"""
You are an expert AI teacher.

Question:
{question}

Answer clearly.
"""
```

This works.

Now imagine 500 users.

Then 5,000 users.

Then multiple AI applications.

Suddenly we are writing:

```python
f"..."
f"..."
f"..."
f"..."
```

everywhere.

This becomes difficult to maintain.

---

# Software Engineering Perspective

Imagine building a website.

Would you hardcode HTML repeatedly?

No.

You use templates.

Example:

```html
Hello {{name}}
```

Then inject data later.

Prompt templates follow the same philosophy.

---

# First-Principles Definition

A Prompt Template is:

> A reusable blueprint for generating prompts.

Instead of:

```python
prompt = f"""
Question:
{question}
"""
```

we create:

```python
template = """
Question:
{question}
"""
```

and fill values later.

---

# Real-World Analogy

Imagine a university degree certificate.

The format remains identical.

Only specific fields change.

```text
Certificate of Graduation

Awarded To:
{student_name}

Degree:
{degree_name}
```

The certificate structure stays constant.

The variables change.

Prompt Templates work the same way.

---

# Why Prompt Templates Matter

Without templates:

```python
prompt1 = ...
prompt2 = ...
prompt3 = ...
```

With templates:

```python
template.format(...)
```

Benefits:

- Reusability
- Maintainability
- Consistency
- Cleaner code

---

# Installing Required Packages

For the remainder of Day 3:

```bash
pip install langchain
pip install langchain-core
pip install langchain-groq
```

---

# Importing PromptTemplate

In [1]:
from langchain_core.prompts import PromptTemplate

---

# Creating Your First Prompt Template

In [3]:
prompt = PromptTemplate.from_template(
    """
    You are an AI teacher.

    Question:
    {question}

    Answer clearly.
    """
)

---

# What Happened Here?

We created:

```text
Template Object
```

not a prompt.

The prompt does not exist yet.

Only a blueprint exists.

---

# Visualizing Internally

Current state:

```text
Template

Question:
{question}
```

Variable:

```text
question
```

still empty.

---

# Rendering The Prompt


In [4]:
result = prompt.invoke(
    {
        "question": "What is a Transformer?"
    }
)

print(result)

text='\n    You are an AI teacher.\n\n    Question:\n    What is a Transformer?\n\n    Answer clearly.\n    '


# Why invoke() Works

Remember Part 2.

Everything is a Runnable.

PromptTemplate is also a Runnable.

Therefore:

```python
prompt.invoke(...)
```

works naturally.

---

# Chat Models

Now we need something that consumes prompts.

That something is a Chat Model.

---

# What Problem Do Chat Models Solve?

Without LangChain:

```python
client.chat.completions.create(...)
```

Different providers expose different APIs.

Example:

OpenAI:

```python
client.chat.completions.create(...)
```

Anthropic:

```python
client.messages.create(...)
```

Groq:

```python
client.chat.completions.create(...)
```

Gemini:

```python
generate_content(...)
```

Different syntax.

Different conventions.

Different response structures.

---

# LangChain's Solution

Wrap every provider behind a common interface.

```python
model.invoke(...)
```

regardless of provider.

---

# Creating A Groq Chat Model


In [5]:
from langchain_groq import ChatGroq
import os

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=os.getenv("GROQ_API_KEY")   
)

---

# Why This Is Powerful

Today:

```python
ChatGroq
```

Tomorrow:

```python
ChatOpenAI
```

Your workflow remains almost unchanged.

---

# Calling The Model


In [6]:
response = llm.invoke(
    "Explain Transformers simply."
)

print(response.content)

**What are Transformers?**
Transformers are a type of artificial intelligence (AI) model used for natural language processing (NLP) tasks. They're primarily designed to handle sequential data, like text or speech.

**Key Components:**

1. **Encoder**: Takes in input data (e.g., text) and converts it into a continuous representation.
2. **Decoder**: Generates output data (e.g., translated text) based on the encoded input.

**How Transformers Work:**

1. **Self-Attention Mechanism**: The model focuses on different parts of the input data and weighs their importance.
2. **Layer Stacking**: Multiple layers of encoders and decoders are stacked to refine the output.
3. **Training**: The model learns to predict the next word or character in a sequence, given the context.

**Transformers are useful for:**

* Machine translation
* Text summarization
* Sentiment analysis
* Chatbots and conversational AI

**In simple terms**, Transformers are powerful AI models that help computers understand and 

# Understanding The Response Object

The model does not return plain text.

It returns an AI message object.

Example:

```python
AIMessage(
    content="Transformers are..."
)
```

To get text:

```python
response.content
```

---

# Visual Flow

```text
Prompt
   │
   ▼
Chat Model
   │
   ▼
AIMessage
```

---

# Combining Prompt + Model

Now things become interesting.

Because both are Runnables.

---

# LCEL Composition

```python
chain = prompt | llm
```

---

# Internal Execution

Equivalent to:

```python
formatted_prompt = prompt.invoke(data)

response = llm.invoke(
    formatted_prompt
)
```


---

# Complete Example

In [7]:
from langchain_core.prompts import PromptTemplate
from langchain_groq import ChatGroq

prompt = PromptTemplate.from_template(
    """
    Explain:

    {topic}
    """
)

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=os.getenv("GROQ_API_KEY")
)

chain = prompt | llm

result = chain.invoke(
    {
        "topic": "Embeddings"
    }
)

print(result.content)

**Embeddings**

Embeddings are a fundamental concept in natural language processing (NLP) and machine learning. They are a way to represent words, phrases, or other types of data as dense vectors in a high-dimensional space. This allows words with similar meanings to be mapped to nearby points in the vector space, enabling machines to understand the relationships between them.

**What are Embeddings?**
------------------------

In traditional NLP approaches, words are represented as one-hot encoded vectors, where each word is assigned a unique index in a vocabulary. However, this approach has several limitations:

*   **Sparsity**: One-hot encoded vectors are sparse, meaning that most of the elements are zero, which can lead to inefficient computation and storage.
*   **Lack of semantic meaning**: One-hot encoded vectors do not capture the semantic relationships between words.

Embeddings address these limitations by representing words as dense vectors in a high-dimensional space. Thes

# Output Parsers

Now we reach the third component.

---

# The Problem Output Parsers Solve

LLMs are unpredictable.

Suppose we ask:

```text
Give me:
name
age
city
```

The model might return:

```text
Name: John
Age: 24
City: Lahore
```

or

```text
John, 24, Lahore
```

or

```json
{
  "name":"John"
}
```

Applications need consistency.

Machines hate ambiguity.

---

# Why Parsers Exist

Parsers convert:

```text
Messy LLM Output
```

into:

```text
Reliable Structured Output
```

---

# First-Principles Definition

An Output Parser is:

> A component that transforms model output into a desired format.

---

# Real-World Analogy

Imagine airport security.

Passengers arrive in many forms.

Security standardizes everything.

Similarly:

```text
LLM Output
      │
      ▼
Parser
      │
      ▼
Standardized Format
```

---

# The Simplest Parser

Import:

In [8]:
from langchain_core.output_parsers import StrOutputParser


# Creating The Parser

In [9]:
parser = StrOutputParser()

---

# What Does It Do?

Input:

```python
AIMessage(
    content="Hello World"
)
```

Output:

```python
"Hello World"
```

It extracts only the text.

---

# Example

In [11]:
parsed = parser.invoke(
    result
)

print(parsed)

**Embeddings**

Embeddings are a fundamental concept in natural language processing (NLP) and machine learning. They are a way to represent words, phrases, or other types of data as dense vectors in a high-dimensional space. This allows words with similar meanings to be mapped to nearby points in the vector space, enabling machines to understand the relationships between them.

**What are Embeddings?**
------------------------

In traditional NLP approaches, words are represented as one-hot encoded vectors, where each word is assigned a unique index in a vocabulary. However, this approach has several limitations:

*   **Sparsity**: One-hot encoded vectors are sparse, meaning that most of the elements are zero, which can lead to inefficient computation and storage.
*   **Lack of semantic meaning**: One-hot encoded vectors do not capture the semantic relationships between words.

Embeddings address these limitations by representing words as dense vectors in a high-dimensional space. Thes

# Full Pipeline

Now all three components connect.

```python
chain = (
    prompt
    | llm
    | parser
)
```

---

# Internal Execution

Equivalent to:


In [ ]:
formatted_prompt = prompt.invoke(
    {
        "topic": "Embeddings"
    }
)

response = llm.invoke(
    formatted_prompt
)

final_text = parser.invoke(
    response
)


---

# Complete Runnable Example


In [12]:
from langchain_core.prompts import PromptTemplate

from langchain_core.output_parsers import (
    StrOutputParser
)

from langchain_groq import ChatGroq

prompt = PromptTemplate.from_template(
    """
    Explain:

    {topic}
    """
)

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=os.getenv("GROQ_API_KEY")
)

parser = StrOutputParser()

chain = (
    prompt
    | llm
    | parser
)

result = chain.invoke(
    {
        "topic": "Transformers"
    }
)

print(result)

**Introduction to Transformers**

Transformers are a type of neural network architecture introduced in the paper "Attention Is All You Need" by Vaswani et al. in 2017. They revolutionized the field of natural language processing (NLP) and have been widely adopted in many other areas of artificial intelligence (AI).

**What is a Transformer?**
------------------------

A transformer is a deep learning model that relies entirely on self-attention mechanisms to process input sequences, such as text or images. Unlike traditional recurrent neural networks (RNNs) or convolutional neural networks (CNNs), transformers do not use recurrent connections or convolutional layers. Instead, they use self-attention to weigh the importance of different input elements relative to each other.

**Key Components of a Transformer**
-----------------------------------

1. **Self-Attention Mechanism**: This is the core component of a transformer. It allows the model to attend to different parts of the input s

---

# Execution Flow Diagram

```text
User Input
      │
      ▼
PromptTemplate
      │
      ▼
Formatted Prompt
      │
      ▼
ChatGroq
      │
      ▼
AIMessage
      │
      ▼
StrOutputParser
      │
      ▼
Plain Text
```

---

# Understanding What We Just Built

This is your first true LangChain workflow.

You now have:

```text
Input
   │
   ▼
Prompt Generation
   │
   ▼
Model Invocation
   │
   ▼
Output Processing
```

which is the fundamental pattern behind:

- RAG Systems
- Agents
- Chatbots
- Tool Calling
- AI Workflows

Everything later becomes a more sophisticated version of this same pipeline.

---

# Common Beginner Mistake #1

Writing prompts manually everywhere.

Use Prompt Templates.

---

# Common Beginner Mistake #2

Thinking models return strings.

They return message objects.

Usually access:

```python
response.content
```

---

# Common Beginner Mistake #3

Skipping parsers.

Applications eventually require structured outputs.

Parsers become essential.

---

# Common Beginner Mistake #4

Forgetting that PromptTemplate is a Runnable.

Many beginners think only models are executable.

Remember:

```text
PromptTemplate
```

also supports:

```python
.invoke()
```

because it is a Runnable.

---

# Interview-Level Questions

## Question 1

Why do Prompt Templates exist?

Answer:

To create reusable prompt blueprints that separate prompt structure from runtime data.

---

## Question 2

Why does LangChain wrap model providers?

Answer:

To provide a consistent interface regardless of the underlying provider.

---

## Question 3

What problem do Output Parsers solve?

Answer:

They transform unpredictable LLM outputs into predictable formats usable by applications.

---

## Question 4

Why can Prompt Templates participate in LCEL chains?

Answer:

Because PromptTemplate implements the Runnable interface.

---

## Question 5

What is the output type of a chat model?

Answer:

Typically an AIMessage object, not a plain string.

---

# Mental Model To Carry Forward

Remember this pipeline:

```text
Prompt Template
       │
       ▼
Chat Model
       │
       ▼
Output Parser
```

Everything else in LangChain is built by extending this idea.

When we add:

```text
Retrievers
Memory
Tools
Agents
```

they simply become additional Runnables inserted into the pipeline.

---


# Day 3 — LangChain Core

## Part 4 — RunnableLambda, RunnablePassthrough, and RunnableParallel

---

# The Moment LangChain Becomes Powerful

Up until now, LangChain may not seem particularly impressive.

We learned:

```python
prompt | llm | parser
```

which is essentially:

```python
step1()
step2()
step3()
```

with nicer syntax.

A reasonable question is:

> Why do we need an entire framework just for that?

The answer is:

You don't.

If all your application does is:

```text
Prompt
  ↓
LLM
  ↓
Output
```

LangChain provides very little value.

The real power begins when workflows become more complex.

---

# The Limitation Of Simple Chains

Consider:

```text
User Question
      │
      ▼
Prompt
      │
      ▼
LLM
      │
      ▼
Output
```

This is linear.

Every step follows the previous one.

But real AI applications rarely look like this.

---

# Example: A RAG Application

A Retrieval-Augmented Generation system requires:

```text
User Question
      │
      ├──────────────┐
      │              │
      ▼              ▼
Question      Vector Search
      │              │
      └──────┬───────┘
             ▼
      Build Context
             │
             ▼
           Prompt
             │
             ▼
             LLM
```

Notice something important.

The same question is needed in multiple places.

A simple chain cannot easily do this.

We need more powerful workflow primitives.

---

# The Three Special Runnables

Today we learn:

1. RunnableLambda
2. RunnablePassthrough
3. RunnableParallel

These are the LEGO blocks of LCEL.

Once you understand them, you can build surprisingly sophisticated workflows.

---

# RunnableLambda

---

# The Problem RunnableLambda Solves

Imagine we have:

```python
prompt | llm | parser
```

Now suppose we want a custom Python function.

Example:

```python
def clean_text(text):
    return text.upper()
```

How do we insert this into the chain?

---

Without RunnableLambda:

```python
result = chain.invoke(data)

result = clean_text(result)
```

The custom function exists outside the workflow.

Not ideal.

---

# Why RunnableLambda Was Invented

LangChain needed a way to turn ordinary Python functions into Runnables.

This would allow:

```python
Python Function
```

to behave exactly like:

```python
PromptTemplate
```

or

```python
ChatModel
```

---

# First-Principles Definition

RunnableLambda is:

> A wrapper that converts a normal Python function into a Runnable.

---

# Visual Representation

Normal Function:

```text
Python Function
```

Wrapped:

```text
RunnableLambda
      │
      ▼
Runnable
```

---

# Import

```python
from langchain_core.runnables import RunnableLambda
```

---

# Example 1

Create a simple function:

```python
def uppercase(text):
    return text.upper()
```

Wrap it:

```python
uppercase_runnable = RunnableLambda(
    uppercase
)
```

Execute:

```python
result = uppercase_runnable.invoke(
    "hello"
)

print(result)
```

Output:

```text
HELLO
```

---

# Internal Execution

When you run:

```python
uppercase_runnable.invoke(
    "hello"
)
```

LangChain does:

```python
uppercase("hello")
```

Nothing magical.

RunnableLambda simply adapts the function to the Runnable interface.

---

# Example 2

Mathematical Function

```python
from langchain_core.runnables import (
    RunnableLambda
)

def square(x):
    return x * x

square_runnable = RunnableLambda(
    square
)

print(
    square_runnable.invoke(5)
)
```

Output:

```text
25
```

---

# Why This Is Important

Now your own functions can participate in LCEL.

Example:

```python
chain = (
    prompt
    | llm
    | parser
    | uppercase_runnable
)
```

Flow:

```text
Prompt
   ▼
LLM
   ▼
Parser
   ▼
Custom Function
```

---

# RunnablePassthrough

---

# The Problem RunnablePassthrough Solves

Consider this situation.

We have:

```text
User Question
```

and later we need:

```text
Original Question
```

again.

Normally:

```python
question
```

would get consumed by the next component.

The original value may be lost.

---

# Real-World Analogy

Imagine a photocopy machine.

You give it:

```text
Original Document
```

It returns:

```text
Original Document
+
Copy
```

Nothing changes.

The data simply passes through.

---

# First-Principles Definition

RunnablePassthrough is:

> A Runnable that returns its input unchanged.

---

# Import

```python
from langchain_core.runnables import (
    RunnablePassthrough
)
```

---

# Example

In [13]:
from langchain_core.runnables import (
    RunnablePassthrough
)

In [14]:
passthrough = RunnablePassthrough()

result = passthrough.invoke(
    "Hello"
)

print(result)

Hello


# Wait... Why Is This Useful?

At first glance:

```python
return input
```

seems pointless.

But it becomes powerful inside larger workflows.

---

# Example: Preserving Original Question

Input:

```python
{
    "question": "What is AI?"
}
```

We may want:

```python
{
    "question": "What is AI?",
    "context": retrieved_docs
}
```

later.

RunnablePassthrough helps preserve original inputs while other operations occur.

---

# RunnableParallel

Now we reach the most important concept in this chapter.

---

# The Problem RunnableParallel Solves

Suppose we have:

```text
Question
```

and want:

```text
Embedding Search
```

and

```text
Metadata Lookup
```

at the same time.

Without parallelism:

```python
step1()

wait

step2()

wait
```

This wastes time.

---

# Real-World Analogy

Imagine ordering food.

Would a restaurant:

```text
Cook Rice
(wait)

Cook Chicken
(wait)

Prepare Salad
```

?

No.

They prepare everything simultaneously.

---

# First-Principles Definition

RunnableParallel is:

> A Runnable that executes multiple Runnables using the same input.

---

# Visual Diagram

```text
Input
  │
  ├──────► Runnable A
  │
  ├──────► Runnable B
  │
  └──────► Runnable C
```

---

# Import

```python
from langchain_core.runnables import (
    RunnableParallel
)
```

---

# Example

Create functions:


In [15]:
def uppercase(text):
    return text.upper()

def lowercase(text):
    return text.lower()

def length(text):
    return len(text)

In [18]:
from langchain_core.runnables import (
    RunnableLambda
)

upper = RunnableLambda(
    uppercase
)

lower = RunnableLambda(
    lowercase
)

length_fn = RunnableLambda(
    length
)

In [19]:
from langchain_core.runnables import (
    RunnableParallel
)
parallel = RunnableParallel(
    upper=upper,
    lower=lower,
    length=length_fn
)

In [20]:
result = parallel.invoke(
    "Hello World"
)

print(result)

{'upper': 'HELLO WORLD', 'lower': 'hello world', 'length': 11}


---

# Internal Execution

Equivalent to:

```python
{
    "upper": uppercase(text),
    "lower": lowercase(text),
    "length": length(text)
}
```

except LangChain manages the orchestration.

---

# Why RunnableParallel Is Essential For RAG

Remember the RAG architecture.

We need:

```text
Question
```

for:

```text
Retriever
```

and

```text
Prompt Builder
```

simultaneously.

RunnableParallel makes this elegant.

---

# Combining Parallel + Passthrough

A very common pattern:

```python
from langchain_core.runnables import (
    RunnableParallel,
    RunnablePassthrough
)
```

---

Example:

```python
chain = RunnableParallel(
    question=RunnablePassthrough(),
    question_length=RunnableLambda(
        lambda x: len(x)
    )
)
```

Input:

```python
"What is AI?"
```

Output:

```python
{
    "question": "What is AI?",
    "question_length": 11
}
```

---

# Why This Matters

One input can generate many outputs.

This becomes critical in:

- RAG
- Agents
- Tool Calling
- Multi-step workflows

---

# Complete Runnable Demonstration


In [21]:
from langchain_core.runnables import (
    RunnableLambda,
    RunnableParallel,
    RunnablePassthrough
)

uppercase = RunnableLambda(
    lambda x: x.upper()
)

word_count = RunnableLambda(
    lambda x: len(x.split())
)

chain = RunnableParallel(
    original=RunnablePassthrough(),
    uppercase=uppercase,
    words=word_count
)

result = chain.invoke(
    "LangChain is powerful"
)

print(result)

{'original': 'LangChain is powerful', 'uppercase': 'LANGCHAIN IS POWERFUL', 'words': 3}


---

# How These Components Power Real Applications

---

## RunnableLambda

Used for:

- Data cleaning
- Data transformation
- Custom logic
- API formatting
- Pre-processing
- Post-processing

---

## RunnablePassthrough

Used for:

- Preserving inputs
- Building dictionaries
- Keeping context available
- RAG pipelines

---

## RunnableParallel

Used for:

- Retrieval pipelines
- Multi-model workflows
- Agent orchestration
- Simultaneous computations

---

# Mapping To Day 2 Concepts

Day 2 ReAct Agent:

```python
question

tool_selection()

tool_execution()

reasoning()
```

Many intermediate values must survive.

Many operations happen simultaneously.

These Runnable primitives make that manageable.

---

# Common Beginner Mistake #1

Using RunnableLambda for everything.

If a built-in LangChain component exists:

use it.

RunnableLambda should be reserved for custom logic.

---

# Common Beginner Mistake #2

Not understanding RunnableParallel outputs.

Remember:

```python
RunnableParallel(...)
```

always returns a dictionary.

---

# Common Beginner Mistake #3

Thinking RunnablePassthrough is useless.

In retrieval systems it becomes one of the most frequently used components.

---

# Common Beginner Mistake #4

Forgetting that these are all Runnables.

Meaning:

```python
.invoke()
```

works on all of them.

---

# Interview-Level Questions

## Question 1

What is RunnableLambda?

Answer:

A wrapper that converts a normal Python function into a Runnable.

---

## Question 2

What problem does RunnablePassthrough solve?

Answer:

It preserves and forwards inputs unchanged so they remain available later in a workflow.

---

## Question 3

What does RunnableParallel do?

Answer:

It executes multiple Runnables using the same input and returns their outputs as a dictionary.

---

## Question 4

Why is RunnableParallel important in RAG systems?

Answer:

Because a single user query often needs to be sent simultaneously to retrievers, prompt builders, metadata systems, and other components.

---

## Question 5

Can RunnableLambda participate in LCEL chains?

Answer:

Yes. Once wrapped, it behaves exactly like any other Runnable.

---


# Mental Model To Carry Forward

Think of these three as workflow superpowers:

```text
RunnableLambda
      │
      ▼
Custom Logic

RunnablePassthrough
      │
      ▼
Preserve Data

RunnableParallel
      │
      ▼
Split Workflow

# Day 3 — LangChain Core

## Part 5 — Building Real Chains

### Sequential Chains, Parallel Chains, and Branching Chains

---

# Why We Need More Than One Type Of Chain

Up until now we have built workflows like:

```text
Input
  │
  ▼
Prompt
  │
  ▼
LLM
  │
  ▼
Parser
```

This is called a linear workflow.

Every step executes in a predetermined order.

But real-world AI systems are rarely that simple.

Sometimes:

- Multiple operations must happen simultaneously
- Different user requests require different workflows
- Results from several components must be merged

This is why chain architectures evolved.

---

# The Three Fundamental Chain Architectures

Almost every AI workflow can be reduced to one of three patterns.

```text
1. Sequential Chain

A → B → C → D
```

```text
2. Parallel Chain

      A
     /|\
    / | \
   B  C  D
```

```text
3. Branching Chain

         Input
            │
      ┌─────┴─────┐
      ▼           ▼
  Route A     Route B
```

Understanding these three architectures is more important than memorizing LangChain APIs.

---

# Part 1 — Sequential Chains

---

# The Problem Sequential Chains Solve

Imagine a workflow:

Step 1:

```text
Generate Summary
```

Step 2:

```text
Translate Summary
```

Step 3:

```text
Improve Translation
```

Each step depends on the previous step.

This is a sequential workflow.

---

# Real-World Analogy

Consider a manufacturing line.

```text
Raw Material
      │
      ▼
Cutting
      │
      ▼
Assembly
      │
      ▼
Painting
      │
      ▼
Inspection
```

You cannot paint before assembly.

You cannot inspect before painting.

The order matters.

---

# First Sequential Chain

Let's create something simple.

Input:

```text
Artificial Intelligence
```

Workflow:

```text
Topic
   │
   ▼
Explain Topic
   │
   ▼
Convert To Bullet Points
```

---

# Complete Code

```python
from langchain_core.prompts import PromptTemplate

from langchain_core.output_parsers import (
    StrOutputParser
)

from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key="YOUR_API_KEY"
)

parser = StrOutputParser()

explanation_prompt = PromptTemplate.from_template(
    """
    Explain the following topic clearly:

    {topic}
    """
)

bullet_prompt = PromptTemplate.from_template(
    """
    Convert the following explanation
    into concise bullet points.

    Explanation:
    {explanation}
    """
)

chain = (
    explanation_prompt
    | llm
    | parser
)
```

---

# Why This Is Not Enough

The second prompt expects:

```python
{
    "explanation": ...
}
```

But the first chain returns:

```python
string
```

We need a transformation step.

---

# Adding RunnableLambda

```python
from langchain_core.runnables import (
    RunnableLambda
)

format_output = RunnableLambda(
    lambda text: {
        "explanation": text
    }
)
```

---

# Final Sequential Workflow

```python
chain = (
    explanation_prompt
    | llm
    | parser
    | format_output
    | bullet_prompt
    | llm
    | parser
)
```

---

# Execution Flow

```text
Topic
  │
  ▼
Explanation Prompt
  │
  ▼
LLM
  │
  ▼
Explanation
  │
  ▼
Format Dictionary
  │
  ▼
Bullet Prompt
  │
  ▼
LLM
  │
  ▼
Final Bullet Points
```

---

# Testing

```python
result = chain.invoke(
    {
        "topic": "Transformers"
    }
)

print(result)
```

---

# What We Learned

Sequential Chains are ideal when:

```text
Output A
      ↓
Input B
      ↓
Output B
      ↓
Input C
```

---

# Part 2 — Parallel Chains

---

# The Problem Parallel Chains Solve

Suppose a user asks:

```text
Explain Artificial Intelligence
```

We want:

```text
1. Simple Explanation
2. Technical Explanation
3. Beginner Analogy
```

All three can be generated independently.

Running them sequentially wastes time.

---

# Real-World Analogy

Imagine a newspaper.

One event occurs.

Different journalists create:

```text
Political Analysis
Economic Analysis
Social Analysis
```

simultaneously.

---

# Building A Parallel Chain

---

# Create Prompts

```python
simple_prompt = PromptTemplate.from_template(
    """
    Explain simply:

    {topic}
    """
)

technical_prompt = PromptTemplate.from_template(
    """
    Give a technical explanation:

    {topic}
    """
)

analogy_prompt = PromptTemplate.from_template(
    """
    Explain using a real-world analogy:

    {topic}
    """
)
```

---

# Create Sub-Chains

```python
simple_chain = (
    simple_prompt
    | llm
    | parser
)

technical_chain = (
    technical_prompt
    | llm
    | parser
)

analogy_chain = (
    analogy_prompt
    | llm
    | parser
)
```

---

# Combine With RunnableParallel

```python
from langchain_core.runnables import (
    RunnableParallel
)

parallel_chain = RunnableParallel(
    simple=simple_chain,
    technical=technical_chain,
    analogy=analogy_chain
)
```

---

# Execute

```python
result = parallel_chain.invoke(
    {
        "topic": "Embeddings"
    }
)

print(result)
```

---

# Output Structure

```python
{
    "simple": "...",
    "technical": "...",
    "analogy": "..."
}
```

---

# Internal Flow

```text
Topic
 │
 ├──────────────► Simple Chain
 │
 ├──────────────► Technical Chain
 │
 └──────────────► Analogy Chain
```

---

# Why This Matters

Parallel workflows appear everywhere.

Examples:

```text
Question
   │
   ├─► Retriever
   │
   ├─► Metadata Search
   │
   ├─► Intent Detection
   │
   └─► Safety Analysis
```

This is one of the foundations of modern AI systems.

---

# Part 3 — Branching Chains

---

# The Problem Branching Chains Solve

Imagine:

User asks:

```text
What is Machine Learning?
```

We should:

```text
Explain Concept
```

---

User asks:

```text
2 + 2
```

We should:

```text
Use Calculator
```

---

User asks:

```text
Weather in Lahore
```

We should:

```text
Use Weather Tool
```

Different inputs require different workflows.

This is branching.

---

# Real-World Analogy

Think of a road intersection.

```text
          Input
             │
             ▼
       Decision Point
        /         \
       /           \
 Route A       Route B
```

The route depends on conditions.

---

# Why Branching Is Important

Without branching:

Every request follows the same path.

With branching:

The workflow adapts.

This is the beginning of agent-like behavior.

---

# Simple Branching Example

---

# Import RunnableBranch

```python
from langchain_core.runnables import (
    RunnableBranch
)
```

---

# Create Routes

Math Route:

```python
math_chain = RunnableLambda(
    lambda x: "Math Query Detected"
)
```

---

General Route:

```python
general_chain = RunnableLambda(
    lambda x: "General Query Detected"
)
```

---

# Create Branch

```python
branch = RunnableBranch(
    (
        lambda x: any(
            char.isdigit()
            for char in x
        ),
        math_chain
    ),
    general_chain
)
```

---

# Test

```python
print(
    branch.invoke(
        "2 + 2"
    )
)
```

Output:

```text
Math Query Detected
```

---

```python
print(
    branch.invoke(
        "Explain AI"
    )
)
```

Output:

```text
General Query Detected
```

---

# Internal Execution

Input:

```text
2 + 2
```

Decision:

```python
contains_digit == True
```

Route:

```text
Math Chain
```

---

Input:

```text
Explain AI
```

Decision:

```python
contains_digit == False
```

Route:

```text
General Chain
```

---

# How Branching Relates To ReAct

Remember Day 2.

Your ReAct loop looked something like:

```python
if weather:
    weather_tool()

elif calculator:
    calculator_tool()

elif search:
    search_tool()
```

That is branching.

LangChain simply formalizes it.

---

# Comparison Of The Three Architectures

| Architecture | Flow | Use Case |
|-------------|--------|-----------|
| Sequential | A → B → C | Multi-step workflows |
| Parallel | A → (B,C,D) | Simultaneous operations |
| Branching | A → B OR C | Routing decisions |

---

# Where These Appear In Production

---

## Sequential

Examples:

```text
Question
   ▼
Retriever
   ▼
Prompt Builder
   ▼
LLM
```

RAG systems.

---

## Parallel

Examples:

```text
Question
   ├─► Intent Detection
   ├─► Retrieval
   ├─► Safety Check
   └─► Metadata Lookup
```

Enterprise assistants.

---

## Branching

Examples:

```text
User Query
      │
      ▼
Router
      │
 ┌────┼────┐
 ▼    ▼    ▼
Math Search Weather
```

Agent systems.

---

# Common Beginner Mistake #1

Trying to force every workflow into a sequential chain.

Many workflows are naturally parallel.

---

# Common Beginner Mistake #2

Using parallel execution when outputs depend on one another.

Dependency implies sequential execution.

---

# Common Beginner Mistake #3

Creating giant branching logic.

When branching becomes complex:

```text
Router
   │
   ▼
Specialized Agents
```

is usually a better design.

---

# Common Beginner Mistake #4

Thinking Branching Equals Agents.

Branching is only one ingredient.

Agents also require:

```text
Reasoning
Memory
Tool Selection
Observation
Planning
```

---

# Interview-Level Questions

## Question 1

What is a Sequential Chain?

Answer:

A workflow where each step depends on the output of the previous step.

---

## Question 2

What is a Parallel Chain?

Answer:

A workflow where multiple operations execute independently using the same input.

---

## Question 3

What is a Branching Chain?

Answer:

A workflow that routes inputs to different execution paths based on conditions.

---

## Question 4

Why are Parallel Chains useful in RAG?

Answer:

Because retrieval, metadata lookup, intent analysis, and safety checks can happen simultaneously.

---

## Question 5

How does Branching relate to ReAct?

Answer:

Tool selection in ReAct is fundamentally a branching decision process.

---

# Mental Model To Carry Forward

Everything you build in AI engineering eventually reduces to:

```text
Sequential Processing
```

or

```text
Parallel Processing
```

or

```text
Conditional Routing
```

Master these three patterns and you will understand the architecture behind:

- LangChain
- LangGraph
- CrewAI
- AutoGen
- OpenAI Agents
- Enterprise RAG Systems



# Day 3 — LangChain Core

# Part 6 — Document Q&A System (RAG) From First Principles

---

# Why This Chapter Matters

Everything we learned so far was preparation.

Prompts.

Models.

Runnables.

Chains.

LCEL.

All of those concepts converge into one of the most important architectures in modern AI:

```text
RAG
```

Retrieval-Augmented Generation.

If you understand RAG deeply, you understand the foundation behind:

- ChatGPT with files
- Enterprise AI assistants
- AI search engines
- Internal company knowledge bots
- Legal document assistants
- Medical knowledge systems
- Research assistants

This chapter is arguably more important than the rest of Day 3 combined.

---

# The Fundamental Problem

Let's imagine we have an LLM.

Example:

```text
Llama
GPT
Claude
Gemini
```

A user asks:

```text
What are the refund policies described in my company handbook?
```

The LLM responds:

```text
I don't know.
```

Why?

Because the handbook is not inside the model.

---

# Understanding The Limitation Of LLMs

An LLM only knows:

```text
1. Training Data
2. Information In Current Context Window
```

It does NOT know:

```text
Your PDF
Your Notes
Your Database
Your Documents
Your Research Papers
```

unless you provide them.

---

# Naive Solution #1

Give the entire document.

Example:

```text
Document:
500 pages

Question:
What is the refund policy?
```

Problem:

```text
Context Window Limit
```

Even modern models eventually run out of space.

---

# Naive Solution #2

Retrain The Model

Fine-tune on company documents.

Problem:

```text
Expensive
Slow
Hard To Update
```

Every document change requires retraining.

Not practical.

---

# The Big Idea Behind RAG

Instead of teaching the model everything permanently:

retrieve relevant information at runtime.

---

# First-Principles Definition

RAG means:

```text
Retrieve Relevant Information
          +
Generate Answer
```

---

# Visual Overview

```text
User Question
       │
       ▼
Retrieve Relevant Chunks
       │
       ▼
Inject Into Prompt
       │
       ▼
LLM Generates Answer
```

The model does not memorize documents.

The model reads them when needed.

---

# Real-World Analogy

Imagine a lawyer.

Does the lawyer memorize every law ever written?

No.

The lawyer:

```text
Receives Question
       │
       ▼
Finds Relevant Documents
       │
       ▼
Reads Relevant Sections
       │
       ▼
Answers Question
```

That is RAG.

---

# The Full Retrieval Pipeline

A complete RAG system contains:

```text
Document
    │
    ▼
Chunking
    │
    ▼
Embeddings
    │
    ▼
Vector Database
    │
    ▼
Retriever
    │
    ▼
Context
    │
    ▼
LLM
```

We will understand every box.

---

# Step 1 — Document Loading

---

# The Problem

Computers cannot reason over:

```text
PDF
TXT
DOCX
```

directly.

The document must first be loaded into memory.

---

# Example

Suppose we have:

```text
ai_notes.txt
```

Contents:

```text
Transformers use self-attention
to process relationships between tokens.
```

We must load it.

---

# LangChain Document Object

Internally LangChain converts files into:

```python
Document(
    page_content="...",
    metadata={}
)
```

Think of a Document object as:

```text
Text
+
Metadata
```

---

# Example

```python
Document(
    page_content="Transformers use self-attention",
    metadata={
        "source": "ai_notes.txt"
    }
)
```

---

# Why Metadata Matters

Metadata can store:

```text
File Name
Page Number
Author
Date
Section
```

Useful for citations later.

---

# Step 2 — Chunking

---

# The Problem

Suppose a document contains:

```text
100,000 words
```

Embedding the entire document as one vector is a terrible idea.

Why?

---

# Imagine This Question

```text
What is self-attention?
```

If the document contains:

```text
Chapter 1
Chapter 2
Chapter 3
Chapter 4
...
```

one giant embedding mixes everything together.

Search quality becomes poor.

---

# Why Chunking Was Invented

We divide documents into smaller pieces.

```text
Document
     │
     ▼
Chunk 1
Chunk 2
Chunk 3
Chunk 4
```

Now retrieval becomes precise.

---

# Real-World Analogy

Imagine searching a textbook.

Would you search:

```text
Entire Book
```

or

```text
Relevant Chapter
```

?

Chunking creates mini chapters.

---

# Example

Original:

```text
Paragraph 1
Paragraph 2
Paragraph 3
Paragraph 4
```

Chunked:

```text
Chunk A

Paragraph 1
Paragraph 2
```

```text
Chunk B

Paragraph 3
Paragraph 4
```

---

# Important Insight

Chunking is one of the biggest determinants of RAG quality.

Bad chunking:

```text
Bad Retrieval
```

Good chunking:

```text
Good Retrieval
```

---

# Step 3 — Embeddings

Now we reach the most important concept.

---

# What Problem Do Embeddings Solve?

Computers do not understand:

```text
Meaning
```

Computers understand:

```text
Numbers
```

If we want semantic search:

```text
AI
Machine Learning
Neural Networks
```

must become numbers.

---

# First-Principles Definition

An embedding is:

> A numerical representation of meaning.

---

# Example

Imagine:

```text
Cat
Dog
Tiger
Lion
```

Embedding Space:

```text
Cat   -> [0.8, 0.3, 0.2]
Dog   -> [0.7, 0.2, 0.4]
Tiger -> [0.9, 0.4, 0.2]
```

Similar concepts occupy nearby regions.

---

# Why This Matters

Question:

```text
Tell me about attention mechanisms
```

Chunk:

```text
Self-attention allows tokens...
```

Different words.

Same meaning.

Embeddings help match them.

---

# Retrieval Is Actually Vector Search

This surprises many beginners.

RAG is not searching text.

RAG is searching vectors.

---

# Visual Representation

```text
Chunk
  │
  ▼
Embedding Model
  │
  ▼
Vector
```

---

# Example

```text
"What is AI?"
```

becomes:

```text
[0.234, 0.782, 0.114, ...]
```

Thousands of dimensions.

---

# Step 4 — Vector Database

---

# The Problem

After generating vectors:

Where do we store them?

A normal database is not optimized for:

```text
Nearest Neighbor Search
```

---

# Why Vector Databases Exist

We need:

```text
Find vectors most similar
to this query vector.
```

efficiently.

---

# Vector Database Examples

| Database | Popularity |
|-----------|-----------|
| Chroma | Very High |
| Pinecone | High |
| Weaviate | High |
| Qdrant | High |
| Milvus | High |

For Day 3 we use:

```text
Chroma
```

because it is beginner-friendly.

---

# What Chroma Stores

```text
Chunk
+
Embedding
+
Metadata
```

---

# Example

```text
Chunk:
"Transformers use self-attention"

Vector:
[0.34,0.77,...]

Metadata:
source=notes.txt
```

---

# Retrieval Phase

Now imagine:

User asks:

```text
What is self-attention?
```

---

# Step 1

Question becomes embedding.

```text
Question
     │
     ▼
Embedding Model
     │
     ▼
Question Vector
```

---

# Step 2

Compare against stored vectors.

```text
Question Vector
      │
      ▼
Vector Database
      │
      ▼
Most Similar Chunks
```

---

# Output

Maybe:

```text
Chunk 12

"Self-attention allows tokens
to attend to one another..."
```

---

# Important Insight

The model has not answered yet.

Retrieval only finds information.

Generation happens later.

---

# Context Injection

Now we combine:

```text
Question
+
Retrieved Chunks
```

into a prompt.

---

# Example Prompt

```text
Use the following context
to answer the question.

Context:

Self-attention allows tokens
to attend to one another...

Question:

What is self-attention?
```

---

# Now The LLM Can Answer

```text
Context
      │
      ▼
Prompt
      │
      ▼
LLM
      │
      ▼
Answer
```

---

# Why RAG Is So Powerful

Without RAG:

```text
LLM Knowledge
=
Training Data
```

With RAG:

```text
LLM Knowledge
=
Training Data
+
Your Documents
```

---

# The Complete Architecture

```text
Document
   │
   ▼
Chunking
   │
   ▼
Embeddings
   │
   ▼
Vector DB
   │
   ▼
Retriever

────────────────────

User Question
   │
   ▼
Question Embedding
   │
   ▼
Retriever
   │
   ▼
Relevant Chunks
   │
   ▼
Prompt
   │
   ▼
LLM
   │
   ▼
Answer
```

---

# Mapping RAG To Concepts Already Learned

---

## Transformers

Generate embeddings.

Generate answers.

---

## Embeddings

Power semantic search.

---

## Runnables

Every stage can become a Runnable.

---

## Chains

RAG is a chain.

---

## ReAct

Retrieval can be viewed as a tool.

---

## Agent Loops

Agents often call retrievers.

---

# Common Beginner Mistake #1

Thinking RAG trains the model.

It does not.

Retrieval happens at runtime.

---

# Common Beginner Mistake #2

Thinking vector databases store documents.

They primarily store embeddings and references.

---

# Common Beginner Mistake #3

Ignoring chunk size.

Chunking quality heavily affects retrieval quality.

---

# Common Beginner Mistake #4

Thinking retrieval equals answering.

Retrieval finds information.

The LLM generates the answer.

---

# Interview-Level Questions

## Question 1

Why was RAG invented?

Answer:

To allow LLMs to use external knowledge without retraining.

---

## Question 2

Why not simply place entire documents in the prompt?

Answer:

Context windows are limited and retrieval is more efficient.

---

## Question 3

What is the purpose of chunking?

Answer:

To create smaller searchable units that improve retrieval precision.

---

## Question 4

What is stored inside a vector database?

Answer:

Embeddings, metadata, and references to source text.

---

## Question 5

What is the role of embeddings in RAG?

Answer:

They convert text into vectors that enable semantic similarity search.

---

# Mental Model To Carry Forward

Remember this sentence:

```text
RAG Is Search Before Generation
```

Everything in a retrieval system exists to answer one question:

```text
How do we find the most relevant information
before asking the LLM to generate a response?
```

Once you understand that, the implementation becomes much easier.



# Day 3 — LangChain Core

# Part 6B — Building a Complete Document Q&A System (RAG)

## Full Implementation From First Principles

---

# Before Writing Any Code

In Part 6A we learned:

```text
Document
    │
    ▼
Chunking
    │
    ▼
Embeddings
    │
    ▼
Vector Database
    │
    ▼
Retriever
    │
    ▼
Context Injection
    │
    ▼
LLM
```

Now we will build this pipeline ourselves.

The goal is not merely to make it work.

The goal is to understand:

```text
What each component does
Why it exists
How data flows through it
```

---

# Final System Architecture

```text
ai_notes.txt
       │
       ▼
Document Loader
       │
       ▼
Text Splitter
       │
       ▼
Embeddings
       │
       ▼
ChromaDB
       │
       ▼
Retriever

────────────────────────

User Question
       │
       ▼
Retriever
       │
       ▼
Relevant Chunks
       │
       ▼
Prompt Template
       │
       ▼
LLM
       │
       ▼
Answer
```

---

# Step 1 — Install Dependencies

```bash
pip install langchain
pip install langchain-core
pip install langchain-community
pip install langchain-groq
pip install chromadb
pip install sentence-transformers
```

---

# Why We Need These Packages

| Package | Purpose |
|----------|----------|
| langchain | Core framework |
| langchain-core | LCEL + Runnables |
| langchain-community | Community integrations |
| langchain-groq | Groq models |
| chromadb | Vector database |
| sentence-transformers | Embedding models |

---

# Step 2 — Create A Sample Text File

Create:

```text
ai_notes.txt
```

---

# Contents

```text
Transformers are deep learning architectures.

Transformers use self-attention to model relationships between tokens.

Embeddings convert text into numerical vectors.

Vector databases store embeddings for efficient retrieval.

Retrieval-Augmented Generation combines retrieval and generation.
```

---

# Why We Need A File

The entire purpose of RAG is:

```text
Answer Questions
Using External Knowledge
```

The file represents external knowledge.

---

# Step 3 — Load The Document

---

# Why We Need A File

The entire purpose of RAG is:

```text
Answer Questions
Using External Knowledge
```

The file represents external knowledge.

---

# Step 3 — Load The Document

---

# Import Loader


In [23]:
from langchain_community.document_loaders import (
    TextLoader
)

C:\Users\HP\AppData\Local\Temp\ipykernel_26408\4013721078.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import (


---

# Load File


In [24]:
loader = TextLoader(
    "ai_notes.txt"
)

documents = loader.load()

---

# Inspect Output


In [25]:
print(documents)


[Document(metadata={'source': 'ai_notes.txt'}, page_content='Transformers are deep learning architectures.\n\nTransformers use self-attention to model relationships between tokens.\n\nEmbeddings convert text into numerical vectors.\n\nVector databases store embeddings for efficient retrieval.\n\nRetrieval-Augmented Generation combines retrieval and generation.')]


---

# What's Happening Internally?

The loader reads:

```text
ai_notes.txt
```

and converts it into:

```python
Document
```

objects.

Remember:

```python
Document
=
Text
+
Metadata
```

---

# Step 4 — Split The Document

---

# Why Splitting Matters

Suppose the document contains:

```text
100 pages
```

Retrieving the entire document is inefficient.

Instead:

```text
Chunk A
Chunk B
Chunk C
Chunk D
```

---

# Import Splitter

In [26]:
from langchain_text_splitters import (
    RecursiveCharacterTextSplitter
)

In [27]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=50
)

---

# What These Parameters Mean

---

## chunk_size

Maximum characters per chunk.

Example:

```python
chunk_size=200
```

means:

```text
No chunk exceeds 200 characters
```

---

## chunk_overlap

Keeps some text shared between chunks.

Example:

```python
Chunk A:
ABCDE

Chunk B:
DEFGH
```

Overlap:

```text
DE
```

---

# Why Overlap Exists

Without overlap:

Important information may be cut in half.

Overlap preserves context.

---

# Split Documents

In [28]:
chunks = splitter.split_documents(
    documents
)

In [29]:
print(len(chunks))

2


In [30]:
print(chunks)


[Document(metadata={'source': 'ai_notes.txt'}, page_content='Transformers are deep learning architectures.\n\nTransformers use self-attention to model relationships between tokens.\n\nEmbeddings convert text into numerical vectors.'), Document(metadata={'source': 'ai_notes.txt'}, page_content='Embeddings convert text into numerical vectors.\n\nVector databases store embeddings for efficient retrieval.\n\nRetrieval-Augmented Generation combines retrieval and generation.')]


In [31]:
print(chunks[0])


page_content='Transformers are deep learning architectures.

Transformers use self-attention to model relationships between tokens.

Embeddings convert text into numerical vectors.' metadata={'source': 'ai_notes.txt'}


In [32]:

print(chunks[1])

page_content='Embeddings convert text into numerical vectors.

Vector databases store embeddings for efficient retrieval.

Retrieval-Augmented Generation combines retrieval and generation.' metadata={'source': 'ai_notes.txt'}


In [33]:

print(
    chunks[0].page_content
)

Transformers are deep learning architectures.

Transformers use self-attention to model relationships between tokens.

Embeddings convert text into numerical vectors.


---

# Data Flow

```text
Document
    │
    ▼
Chunk 1
Chunk 2
Chunk 3
Chunk 4
```

---

# Step 5 — Create Embeddings

---

# What Happens Here?

We convert:

```text
Human Language
```

into:

```text
Vectors
```

for semantic search.

---

# Import Embedding Model

In [34]:
from langchain_community.embeddings import (
    HuggingFaceEmbeddings
)

In [35]:
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

C:\Users\HP\AppData\Local\Temp\ipykernel_26408\4145769993.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [36]:
print(embeddings) # the output it produces in this case is the object itself, not the actual embeddings. To get the embeddings, you would need to call the embed method on the text you want to embed. For example: 

client=SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({})
) model_name='all-MiniLM-L6-v2' cache_folder=None model_kwargs={} encode_kwargs={} multi_process=False show_progress=False


---

# Why This Model?

It is:

- Small
- Fast
- Free
- Excellent for learning RAG

---

# Example

Input:

```text
What is AI?
```

Output:

```text
[0.134, 0.876, 0.221 ...]
```

384 dimensions.

---

# Step 6 — Create Chroma Vector Database

---

# Import Chroma


In [37]:
from langchain_community.vectorstores import (
    Chroma
)

In [38]:
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings
)

---

# What Happens Internally?

For every chunk:

```text
Chunk
   │
   ▼
Embedding Model
   │
   ▼
Vector
```

Then Chroma stores:

```text
Chunk
+
Vector
+
Metadata
```

---

# Visual Representation

```text
Chunk A
   │
   ▼
Vector A

Chunk B
   │
   ▼
Vector B

Chunk C
   │
   ▼
Vector C
```

stored inside Chroma.

---

# Step 7 — Create Retriever

---

# Why We Need A Retriever

The vector database stores information.

Something must search it.

That component is:

```text
Retriever
```

---

# Create Retriever


In [39]:
retriever = vector_store.as_retriever(
    search_kwargs={
        "k": 3
    }
)

---

# What Does k Mean?

```python
k=3
```

means:

```text
Return Top 3 Most Similar Chunks
```

---

# Test Retrieval


In [40]:
results = retriever.invoke(
    "What are embeddings?"
)

In [41]:
for doc in results:
    print(doc.page_content)

Embeddings convert text into numerical vectors.

Vector databases store embeddings for efficient retrieval.

Retrieval-Augmented Generation combines retrieval and generation.
Transformers are deep learning architectures.

Transformers use self-attention to model relationships between tokens.

Embeddings convert text into numerical vectors.


---

# What's Happening?

Question:

```text
What are embeddings?
```

↓

Embedding

↓

Vector Search

↓

Most Similar Chunks

---

# Important Insight

The LLM has not been used yet.

This is retrieval only.

---

# Step 8 — Create Prompt Template

Now we must inject retrieved context.

---

# Import

In [42]:
from langchain_core.prompts import (
    PromptTemplate
)

# Create Prompt


In [43]:
prompt = PromptTemplate.from_template(
    """
    Use the context below
    to answer the question.

    Context:

    {context}

    Question:

    {question}

    Answer:
    """
)

---

# Why Context Injection Works

Without context:

```text
LLM doesn't know your file.
```

With context:

```text
LLM can read relevant chunks.
```

---

# Step 9 — Create Groq Model

---


In [44]:
from langchain_groq import ChatGroq

In [46]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=os.getenv("GROQ_API_KEY")
)

In [47]:
from langchain_core.output_parsers import (
    StrOutputParser
)

In [48]:
parser = StrOutputParser()


# Why We Need It

Convert:

```python
AIMessage
```

into:

```python
String
```

---

# Step 11 — Build Retrieval Pipeline

In [49]:
from langchain_core.runnables import (
    RunnablePassthrough
)

In [50]:
rag_chain = (
    {
        "context": retriever,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | parser
)

---

# This Is The Most Important Code So Far

Let's understand it carefully.

---

# Input

```python
"What are embeddings?"
```

---

# RunnablePassthrough

Produces:

```python
{
    "question":
    "What are embeddings?"
}
```

---

# Retriever

Receives:

```python
"What are embeddings?"
```

and returns:

```text
Relevant Chunks
```

---

# Combined Dictionary

```python
{
    "question":
    "What are embeddings?",

    "context":
    retrieved_chunks
}
```

---

# Prompt Template

Produces:

```text
Context:
...

Question:
What are embeddings?
```

---

# LLM

Generates answer.

---

# Parser

Returns plain text.

---

# Final Flow

```text
Question
    │
    ├──────────────┐
    │              │
    ▼              ▼
Retriever      Passthrough
    │              │
    └──────┬───────┘
           ▼
     Prompt Template
           │
           ▼
          LLM
           │
           ▼
        Parser
           │
           ▼
         Answer
```

---

# Step 12 — Ask Questions

---

In [51]:
response = rag_chain.invoke(
    "What are transformers?"
)

print(response)

Transformers are deep learning architectures that use self-attention to model relationships between tokens.


# Full Production Example


In [52]:
from langchain_community.document_loaders import TextLoader

from langchain_text_splitters import (
    RecursiveCharacterTextSplitter
)

from langchain_community.embeddings import (
    HuggingFaceEmbeddings
)

from langchain_community.vectorstores import (
    Chroma
)

from langchain_core.prompts import (
    PromptTemplate
)

from langchain_core.output_parsers import (
    StrOutputParser
)

from langchain_core.runnables import (
    RunnablePassthrough
)

from langchain_groq import ChatGroq


loader = TextLoader(
    "ai_notes.txt"
)

documents = loader.load()


splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=50
)

chunks = splitter.split_documents(
    documents
)


embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings
)

retriever = vector_store.as_retriever(
    search_kwargs={"k": 3}
)

prompt = PromptTemplate.from_template(
    """
    Use the context below
    to answer the question.

    Context:
    {context}

    Question:
    {question}

    Answer:
    """
)

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=os.getenv("GROQ_API_KEY")
)

parser = StrOutputParser()

rag_chain = (
    {
        "context": retriever,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | parser
)

while True:

    query = input(
        "\nAsk: "
    )

    if query.lower() == "exit":
        break

    answer = rag_chain.invoke(
        query
    )

    print("\nAnswer:")
    print(answer)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


Answer:
It appears you haven't provided a question for me to answer based on the given context. Please provide the question, and I'll be happy to assist you using the information provided in the context about deep learning architectures, specifically transformers, embeddings, and vector databases.

Answer:
It appears there is no question provided for me to answer based on the given context. Please provide the question you would like me to answer.

Answer:
It appears you haven't provided a question for me to answer based on the given context. Please provide the question you need help with.

Answer:
It seems like you've provided a context and a question that doesn't relate to the context. The context appears to be about deep learning architectures, specifically transformers and embeddings, while the question is simply "hello world." 

If you're looking for information related to the context, I can tell you that transformers are indeed deep learning architectures that use self-attention 

# Common Beginner Mistake #1

Thinking retrieval happens after generation.

Correct order:

```text
Retrieve
   ↓
Generate
```

---

# Common Beginner Mistake #2

Using huge chunk sizes.

Large chunks:

```text
Poor Precision
```

---

# Common Beginner Mistake #3

Using tiny chunk sizes.

Tiny chunks:

```text
Loss Of Context
```

---

# Common Beginner Mistake #4

Forgetting overlap.

Overlap often improves retrieval quality significantly.

---

# Common Beginner Mistake #5

Thinking Chroma stores text only.

It stores:

```text
Text
+
Embeddings
+
Metadata
```

---

# Interview-Level Questions

## Question 1

Why do we chunk documents?

Answer:

To improve retrieval precision and embedding quality.

---

## Question 2

Why use embeddings instead of keyword search?

Answer:

Embeddings enable semantic similarity rather than exact word matching.

---

## Question 3

What does a retriever do?

Answer:

It converts a query into a vector search and returns relevant chunks.

---

## Question 4

What role does RunnablePassthrough play in RAG?

Answer:

It preserves the original user question while retrieval occurs.

---

## Question 5

What is the most important idea in RAG?

Answer:

Retrieve relevant knowledge before generation.

---

# End of Day 3 — Part 6B

At this point you have completed:

```text
✓ LangChain Architecture
✓ LCEL
✓ Runnables
✓ RunnableLambda
✓ RunnablePassthrough
✓ RunnableParallel
✓ Prompt Templates
✓ Chat Models
✓ Output Parsers
✓ Sequential Chains
✓ Parallel Chains
✓ Branching Chains
✓ Full RAG System
✓ ChromaDB
✓ Retrieval Pipeline

# Day 3 — LangChain Core

# Part 7 — Deep Dive Into RAG Engineering

## From "I Can Build RAG" To "I Understand RAG"

---

# Why This Chapter Exists

Most tutorials stop after building a simple RAG system.

You load:

```text
PDF
  ↓
Chunks
  ↓
Embeddings
  ↓
Vector DB
  ↓
Retriever
  ↓
LLM
```

The system works.

Everyone celebrates.

Then reality arrives.

Users ask:

```text
Why did retrieval miss the answer?

Why did it retrieve irrelevant chunks?

Why are responses hallucinating?

Why is performance inconsistent?

Why is retrieval slow?
```

At that moment you stop being a LangChain user and become an AI engineer.

This chapter is about those problems.

---

# The Biggest Beginner Misconception

Many people believe:

```text
Good Model
=
Good RAG
```

False.

In practice:

```text
Retrieval Quality
>
Model Quality
```

for many enterprise systems.

A powerful model with bad retrieval:

```text
Bad Answers
```

A smaller model with excellent retrieval:

```text
Often Great Answers
```

---

# The RAG Equation

Think of RAG as:

```text
Answer Quality

≈

Retrieval Quality
+
Prompt Quality
+
Model Quality
```

If retrieval fails:

everything downstream suffers.

---

# Section 1 — Chunking Strategies

---

# Why Chunking Is Harder Than It Looks

Most beginners use:

```python
chunk_size=1000
chunk_overlap=200
```

because a tutorial told them to.

This is dangerous.

Chunking is not about numbers.

Chunking is about meaning.

---

# Example Of Bad Chunking

Document:

```text
What is Self-Attention?

Self-attention allows tokens
to attend to other tokens.

Benefits:

1. Parallelization
2. Long-range dependencies
```

Bad split:

```text
Chunk A

What is Self-Attention?

Self-attention allows
```

```text
Chunk B

tokens to attend
to other tokens.

Benefits:
```

Meaning has been destroyed.

---

# The Core Principle

A chunk should represent:

```text
One Coherent Idea
```

not an arbitrary character count.

---

# Common Chunking Strategies

---

## Strategy 1 — Fixed Size Chunking

Example:

```python
RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)
```

Produces:

```text
500 Characters
500 Characters
500 Characters
```

---

### Advantages

```text
Simple
Fast
Predictable
```

---

### Disadvantages

```text
Can break meaning
```

---

# Strategy 2 — Paragraph Chunking

Split on:

```text
Paragraph Boundaries
```

Example:

```text
Paragraph A

Paragraph B

Paragraph C
```

---

### Advantages

Preserves semantic structure.

---

### Disadvantages

Chunk sizes vary significantly.

---

# Strategy 3 — Section Chunking

Split using:

```text
Headings
Subheadings
Sections
```

Example:

```text
Chapter 1

Chapter 2

Chapter 3
```

---

### Advantages

Excellent for:

- textbooks
- documentation
- manuals

---

### Disadvantages

Requires structured documents.

---

# Strategy 4 — Semantic Chunking

Most advanced.

Instead of counting characters:

```text
Split based on meaning.
```

Example:

```text
Topic A

Topic B

Topic C
```

even if sizes differ.

---

### Why Semantic Chunking Exists

Meaning matters more than size.

This often produces better retrieval.

---

# Interview Question

Why is chunking important?

Answer:

Because retrieval quality depends heavily on how information is divided before embedding.

---

# Section 2 — Choosing Embedding Models

---

# The Hidden Truth

Many beginners obsess over:

```text
GPT-4
Claude
Llama
```

while ignoring embeddings.

This is a mistake.

Embeddings drive retrieval.

---

# What Makes A Good Embedding Model?

A model should place semantically similar concepts near one another.

---

Example:

```text
Car
Vehicle
Automobile
```

should occupy nearby vector regions.

---

# Popular Embedding Models

| Model | Use Case |
|---------|---------|
| all-MiniLM-L6-v2 | Learning / Small Projects |
| bge-small | Lightweight Production |
| bge-large | High Accuracy |
| e5-large | Strong Retrieval |
| OpenAI Embeddings | Managed API |
| Voyage Embeddings | State-of-the-art Retrieval |

---

# Tradeoffs

---

## Small Models

Example:

```text
MiniLM
```

Advantages:

```text
Fast
Cheap
```

Disadvantages:

```text
Lower Accuracy
```

---

## Large Models

Advantages:

```text
Better Semantic Search
```

Disadvantages:

```text
More Compute
More Memory
```

---

# Industry Insight

Many production systems spend more effort selecting embeddings than selecting LLMs.

---

# Section 3 — Similarity Search

---

# What Actually Happens During Retrieval?

Many people imagine:

```text
Question
    ↓
Database
    ↓
Answer
```

This is incorrect.

---

Actual process:

```text
Question
    ↓
Embedding
    ↓
Vector
    ↓
Similarity Search
    ↓
Nearest Chunks
```

---

# The Geometry Behind RAG

Imagine a giant space.

```text
AI
Machine Learning
Transformers
Embeddings
```

all occupy positions.

Question:

```text
What are embeddings?
```

gets mapped into this space.

Nearby chunks are retrieved.

---

# Similarity Metrics

---

## Cosine Similarity

Most common.

Measures:

```text
Angle Between Vectors
```

not distance.

---

Example:

```text
Same Direction
=
High Similarity
```

---

## Euclidean Distance

Measures:

```text
Physical Distance
```

between vectors.

---

## Dot Product

Frequently used in large-scale retrieval systems.

---

# Interview Question

Why is cosine similarity popular?

Answer:

Because semantic meaning often corresponds better to vector direction than raw magnitude.

---

# Section 4 — Metadata Filtering

---

# The Problem

Imagine a vector database containing:

```text
2022 Policies
2023 Policies
2024 Policies
```

Question:

```text
What is the refund policy?
```

Retrieval may return outdated documents.

---

# Solution

Metadata Filtering.

---

# Example Metadata

```python
{
    "year": 2024,
    "department": "HR"
}
```

---

# Retrieval With Filters

```python
retriever = vector_store.as_retriever(
    search_kwargs={
        "filter": {
            "year": 2024
        }
    }
)
```

---

# Why This Matters

Semantic search alone is often insufficient.

Metadata provides additional constraints.

---

# Section 5 — Reranking

---

# The Retrieval Problem

Retriever returns:

```text
Top 10 Chunks
```

But ranking may not be ideal.

---

Example:

```text
Chunk 3
```

might actually be better than:

```text
Chunk 1
```

---

# What Reranking Does

Second model evaluates retrieved chunks.

---

Pipeline:

```text
Question
   │
   ▼
Retriever
   │
   ▼
Top 10 Chunks
   │
   ▼
Reranker
   │
   ▼
Best 3 Chunks
```

---

# Why Reranking Works

Retrievers optimize:

```text
Similarity
```

Rerankers optimize:

```text
Relevance
```

These are not always identical.

---

# Industry Insight

Many enterprise systems achieve major improvements simply by adding reranking.

---

# Section 6 — Hybrid Search

---

# The Weakness Of Pure Vector Search

Suppose user asks:

```text
What is RFC 2616?
```

Vector search may struggle.

---

Because:

```text
RFC 2616
```

is a specific keyword.

---

# Keyword Search Excels Here

Example:

```text
RFC 2616
```

exactly matches.

---

# Hybrid Search Combines Both

```text
Keyword Search
        +
Vector Search
```

---

Architecture:

```text
Question
    │
 ┌──┴──┐
 ▼     ▼

BM25  Vector Search
 │      │
 └──┬───┘
    ▼
 Merge Results
```

---

# Why Hybrid Search Is Powerful

Vector Search:

```text
Understands Meaning
```

Keyword Search:

```text
Finds Exact Terms
```

Together they outperform either method alone.

---

# Section 7 — Production RAG Architecture

---

# Beginner Architecture

```text
Document
    ↓
Chunks
    ↓
Retriever
    ↓
LLM
```

Works.

But not production-ready.

---

# Production Architecture

```text
Documents
      │
      ▼
Preprocessing
      │
      ▼
Chunking
      │
      ▼
Embeddings
      │
      ▼
Vector Store
      │
      ▼
Retriever
      │
      ▼
Reranker
      │
      ▼
Prompt Builder
      │
      ▼
LLM
      │
      ▼
Answer
```

---

# Additional Components

Often includes:

```text
Caching
Monitoring
Guardrails
Evaluation
Analytics
Feedback Loops
```

---

# Why Production Is Hard

The challenge is no longer:

```text
Can the system answer?
```

The challenge becomes:

```text
Can it answer reliably
for thousands of users?
```

---

# Section 8 — Interview-Level System Design

---

# Common Interview Question

Design an AI assistant for company documents.

---

# Strong Answer

```text
1. Load Documents

2. Chunk Documents

3. Generate Embeddings

4. Store In Vector DB

5. Create Retriever

6. Add Metadata Filters

7. Add Reranking

8. Inject Context

9. Generate Response

10. Cite Sources
```

---

# Follow-Up Question

What if retrieval quality is poor?

Strong Answer:

```text
Improve Chunking

Improve Embeddings

Increase k

Add Reranking

Use Hybrid Search

Improve Metadata
```

---

# Common Beginner Mistake #1

Assuming bigger chunks are always better.

Large chunks often reduce retrieval precision.

---

# Common Beginner Mistake #2

Assuming smaller chunks are always better.

Tiny chunks lose context.

---

# Common Beginner Mistake #3

Ignoring embedding quality.

Embeddings often determine retrieval success.

---

# Common Beginner Mistake #4

Thinking retrieval and reranking are identical.

They optimize different objectives.

---

# Common Beginner Mistake #5

Believing a tutorial RAG system is production-ready.

Real systems require:

```text
Evaluation
Monitoring
Caching
Reranking
Security
```

---

# Interview-Level Questions

## Question 1

What component most influences retrieval quality?

Answer:

Chunking strategy and embedding quality.

---

## Question 2

Why add reranking?

Answer:

To improve relevance after initial retrieval.

---

## Question 3

Why use metadata filters?

Answer:

To constrain retrieval to relevant subsets of data.

---

## Question 4

What problem does hybrid search solve?

Answer:

It combines semantic search and exact keyword matching.

---

## Question 5

What is the difference between retrieval and reranking?

Answer:

Retrieval finds candidate chunks; reranking reorders them by relevance.

---

# Mental Model To Carry Forward

A beginner sees RAG as:

```text
Retriever → LLM
```

An engineer sees RAG as:

```text
Chunking
   +
Embeddings
   +
Vector Search
   +
Filtering
   +
Reranking
   +
Prompt Design
   +
Evaluation
```

The LLM is only one component.

The real engineering challenge is ensuring the right information reaches the model.

# Day 3 — Part 8A

# Capstone Project Planning

## Building a Python Documentation AI Assistant

---

# Before Writing Code

Most tutorials do:

```text
PDF
 ↓
Chunk
 ↓
Embed
 ↓
Chat
```

and call it RAG.

We are going to build something much closer to a real-world ingestion pipeline.

---

# Final Project Architecture

```text
Python Documentation Website
            │
            ▼
      scrape_docs.py
            │
            ▼
        raw_docs/
            │
            ▼
      clean_docs.py
            │
            ▼
       clean_docs/
            │
            ▼
       build_db.py
            │
            ▼
       chroma_db/
            │
            ▼
   debug_retrieval.py
            │
            ▼
          chat.py
```

---

# Why This Architecture?

Think about a company.

Suppose Microsoft has:

```text
100,000 pages
```

of documentation.

Would they do:

```text
Website
  ↓
Vector DB
```

directly?

No.

Because:

```text
Websites contain noise.
```

Example:

```html
Search

Previous

Next

Navigation

Copyright

Terms Of Service
```

These have no retrieval value.

---

# Therefore We Separate

```text
Raw Data
```

from

```text
Clean Data
```

---

# Project Structure

Create:

```text
python_rag/

│
├── raw_docs/
│
├── clean_docs/
│
├── chroma_db/
│
├── scrape_docs.py
│
├── clean_docs.py
│
├── build_db.py
│
├── debug_retrieval.py
│
├── chat.py
│
└── requirements.txt
```

---

# Step 1

Create Virtual Environment

```bash
python -m venv venv
```

Activate:

Windows:

```bash
venv\Scripts\activate
```

Linux/Mac:

```bash
source venv/bin/activate
```

---

# Why Virtual Environments Exist

Without them:

```text
Project A
and
Project B
```

share dependencies.

Conflicts occur.

Virtual environments isolate projects.

---

# Step 2

Create requirements.txt

```text
langchain
langchain-core
langchain-community
langchain-groq

chromadb

sentence-transformers

beautifulsoup4

requests

lxml

rich
```

---

# Install Everything

```bash
pip install -r requirements.txt
```

---

# Why Each Package Exists

| Package | Purpose |
|----------|----------|
| requests | Download web pages |
| beautifulsoup4 | Parse HTML |
| lxml | Fast HTML parser |
| chromadb | Vector database |
| sentence-transformers | Embeddings |
| rich | Better terminal UI |
| langchain | Framework |

---

# Step 3

Select Documentation Pages

We intentionally choose a small but meaningful corpus.

---

# Corpus

```python
DOC_URLS = [

"https://docs.python.org/3/tutorial/classes.html",

"https://docs.python.org/3/tutorial/modules.html",

"https://docs.python.org/3/tutorial/datastructures.html",

"https://docs.python.org/3/tutorial/inputoutput.html",

"https://docs.python.org/3/library/pathlib.html",

"https://docs.python.org/3/library/json.html",

"https://docs.python.org/3/library/os.html",

"https://docs.python.org/3/library/datetime.html",

"https://docs.python.org/3/library/functions.html",

"https://docs.python.org/3/library/stdtypes.html",

]
```

---

# Why Not Crawl Entire Website?

Let's reason.

Entire Python Docs:

```text
Thousands of pages
```

Embedding:

```text
Hours
```

Storage:

```text
Huge
```

Learning Value:

```text
Not Much More Initially
```

---

# Good Engineering Principle

Start with:

```text
Small Corpus
```

then scale.

---

# Step 4

Build The Scraper

But before coding, let's think.

---

# What Should The Scraper Do?

Input:

```text
URL
```

Output:

```text
HTML File
```

stored in:

```text
raw_docs/
```

---

# Example

Input:

```text
https://docs.python.org/3/library/json.html
```

Output:

```text
raw_docs/json.html
```

---

# Why Save Raw HTML?

Because:

```text
Scraping
```

and

```text
Cleaning
```

should be separate processes.

---

# Imagine This Happens

Your cleaning logic is wrong.

If you only kept processed text:

```text
Must scrape again.
```

---

# With Raw HTML

You can:

```text
Re-clean
Re-process
Re-chunk
```

without hitting the website again.

---

# Data Lifecycle

```text
Website
   │
   ▼
Raw HTML
   │
   ▼
Clean Text
   │
   ▼
Chunks
   │
   ▼
Embeddings
```

---

# Step 5

What HTML Should We Extract?

This is where many beginners make mistakes.

---

# Example HTML Page

```html
Navigation

Search

Previous

Next

Article Content

Footer
```

---

# What Do We Actually Want?

Only:

```text
Article Content
```

---

# Therefore Our Cleaner Must Extract

```text
Headings

Paragraphs

Lists

Code Examples
```

---

# Ignore

```text
Navigation

Search

Footer

Sidebar
```

---

# Why?

Because every useless sentence becomes:

```text
Chunk
```

and pollutes retrieval.

---

# Important Question

Should We Build

```text
scrape_docs.py
```

first

or

```text
build_db.py
```

first?

---

# Think Like An Engineer

Can we build embeddings without documents?

No.

Can we build retrieval without embeddings?

No.

Can we build chat without retrieval?

No.

---

# Dependency Graph

```text
scrape_docs.py
      │
      ▼
clean_docs.py
      │
      ▼
build_db.py
      │
      ▼
debug_retrieval.py
      │
      ▼
chat.py
```

---

# Therefore Next Step

We build:

```text
scrape_docs.py
```

first.

---

# What scrape_docs.py Must Accomplish

1. Visit each URL

2. Download HTML

3. Save locally

4. Skip duplicates

5. Handle failures

6. Show progress

---

# Example Output

```text
Downloading:

classes.html

✓ Saved

modules.html

✓ Saved

json.html

✓ Saved
```

---

# What We Will Learn From This Stage

Not RAG.

Not embeddings.

Not retrieval.

---

We learn:

```text
Data Ingestion
```

which is actually the first stage of every production AI system.


# Day 3 — Part 8B

# Building `scrape_docs.py`

## Data Ingestion Pipeline From First Principles

---

# Before Writing Any Code

Let's think like engineers.

We want:

```text
Python Documentation Website
          │
          ▼
      Local Files
```

Question:

```text
How does a Python program download a webpage?
```

Before learning scraping, you must understand what actually happens.

---

# What Happens When You Open A Website?

You type:

```text
https://docs.python.org/3/library/json.html
```

into your browser.

Your browser does:

```text
Request Page
      │
      ▼
Server Receives Request
      │
      ▼
Server Sends HTML
      │
      ▼
Browser Renders HTML
```

---

# Important Realization

A website is often just:

```text
HTML
+
CSS
+
JavaScript
```

Our RAG system only cares about:

```text
HTML Content
```

---

# What We Need

A program that can:

```text
Send Request
     ↓
Receive HTML
     ↓
Save HTML
```

This is exactly what the `requests` library does.

---

# Why Not Use Selenium?

You may have heard of:

```text
Selenium
Playwright
Puppeteer
```

These launch actual browsers.

Example:

```text
Open Chrome
Load Website
Click Buttons
```

---

# Do We Need Them?

For Python docs:

```text
No
```

Reason:

Python documentation pages are mostly static HTML.

A simple HTTP request is enough.

---

# First HTTP Request

Create:

```python
import requests

url = "https://docs.python.org/3/library/json.html"

response = requests.get(url)

print(response.status_code)
```

---

# What Is Happening?

Step 1:

```python
requests.get(url)
```

sends:

```text
HTTP GET Request
```

to the server.

---

# Server Response

Server sends back:

```text
HTML Content
```

plus metadata.

---

# Status Codes

These are extremely important.

---

## 200

```text
Success
```

---

## 404

```text
Page Not Found
```

---

## 500

```text
Server Error
```

---

## 403

```text
Access Forbidden
```

---

# Example

```python
if response.status_code == 200:
    print("Success")
```

---

# Viewing Raw HTML

```python
print(response.text[:1000])
```

Output:

```html
<!DOCTYPE html>

<html>

<head>
...
```

---

# What Is `response.text`?

It contains:

```text
Entire HTML Document
```

as a Python string.

---

# Saving HTML To Disk

Suppose we downloaded:

```text
json.html
```

We want:

```text
raw_docs/json.html
```

---

# Code

```python
with open(
    "raw_docs/json.html",
    "w",
    encoding="utf-8"
) as file:

    file.write(response.text)
```

---

# Why Use UTF-8?

Documentation contains:

```text
Special Characters
Unicode Symbols
```

UTF-8 handles them safely.

---

# Understanding Context Managers

Many beginners copy:

```python
with open(...)
```

without understanding it.

---

# Without Context Manager

```python
file = open("test.txt")

file.write("hello")

file.close()
```

---

Problem:

If an error occurs:

```python
file.close()
```

might never execute.

---

# With Context Manager

```python
with open(...) as file:
```

Python automatically closes the file.

Safer.

Cleaner.

Professional.

---

# Designing Our Scraper

Before coding, define requirements.

---

# Input

```python
URLS
```

---

# Processing

```text
Download HTML
```

---

# Output

```text
raw_docs/*.html
```

---

# Example

Input:

```python
"https://docs.python.org/3/library/json.html"
```

Output:

```text
raw_docs/json.html
```

---

# How Do We Determine Filename?

Consider:

```text
https://docs.python.org/3/library/json.html
```

---

We only want:

```text
json.html
```

---

# Python Solution

```python
url.split("/")
```

Result:

```python
[
 "https:",
 "",
 "docs.python.org",
 "3",
 "library",
 "json.html"
]
```

---

Last element:

```python
url.split("/")[-1]
```

returns:

```text
json.html
```

---

# Test

```python
url = "https://docs.python.org/3/library/json.html"

filename = url.split("/")[-1]

print(filename)
```

Output:

```text
json.html
```

---

# Building The Download Function

Instead of writing everything in one giant script:

Create reusable functions.

---

# Why Functions?

Bad:

```python
500 lines
no structure
```

Good:

```python
download_page()

save_html()

main()
```

---

# First Function

```python
import requests

def download_page(url):

    response = requests.get(url)

    return response.text
```

---

# Problem With This Version

What if:

```text
Internet fails?
```

or

```text
404 occurs?
```

We need error handling.

---

# Better Version

```python
import requests

def download_page(url):

    try:

        response = requests.get(
            url,
            timeout=10
        )

        response.raise_for_status()

        return response.text

    except Exception as error:

        print(
            f"Failed: {url}"
        )

        print(error)

        return None
```

---

# Understanding `raise_for_status()`

Without:

```python
404 page
```

still returns HTML.

You might save an error page accidentally.

---

With:

```python
response.raise_for_status()
```

Python raises an exception.

Safer.

---

# Saving HTML

Create another function.

---

```python
def save_html(
    html,
    filename
):

    with open(
        filename,
        "w",
        encoding="utf-8"
    ) as file:

        file.write(html)
```

---

# Why Separate Functions?

Because:

```text
Downloading
```

and

```text
Saving
```

are different responsibilities.

This is called:

```text
Separation Of Concerns
```

---

# Full Workflow

```text
URL
 │
 ▼
Download HTML
 │
 ▼
Save HTML
```

---

# Complete Minimal Scraper

```python
import requests

DOC_URLS = [

    "https://docs.python.org/3/tutorial/classes.html",

    "https://docs.python.org/3/tutorial/modules.html",

    "https://docs.python.org/3/library/json.html"
]

for url in DOC_URLS:

    response = requests.get(url)

    filename = url.split("/")[-1]

    with open(
        f"raw_docs/{filename}",
        "w",
        encoding="utf-8"
    ) as file:

        file.write(response.text)

    print(
        f"Saved {filename}"
    )
```

---

# Why This Is Still Not Production Quality

Missing:

```text
Error Handling
Progress Tracking
Duplicate Checks
Timeout Handling
Retries
```

---

# Production Version

```python
import requests

from pathlib import Path

RAW_DIR = Path("raw_docs")

RAW_DIR.mkdir(
    exist_ok=True
)

DOC_URLS = [

    "https://docs.python.org/3/tutorial/classes.html",

    "https://docs.python.org/3/tutorial/modules.html",

    "https://docs.python.org/3/tutorial/datastructures.html",

    "https://docs.python.org/3/tutorial/inputoutput.html",

    "https://docs.python.org/3/library/pathlib.html",

    "https://docs.python.org/3/library/json.html",

    "https://docs.python.org/3/library/os.html",

    "https://docs.python.org/3/library/datetime.html",

    "https://docs.python.org/3/library/functions.html",

    "https://docs.python.org/3/library/stdtypes.html"
]


def download_page(url):

    try:

        response = requests.get(
            url,
            timeout=10
        )

        response.raise_for_status()

        return response.text

    except Exception as error:

        print(
            f"\nFailed: {url}"
        )

        print(error)

        return None


for url in DOC_URLS:

    filename = url.split("/")[-1]

    filepath = (
        RAW_DIR / filename
    )

    if filepath.exists():

        print(
            f"Skipping {filename}"
        )

        continue

    print(
        f"Downloading {filename}"
    )

    html = download_page(url)

    if html is None:
        continue

    with open(
        filepath,
        "w",
        encoding="utf-8"
    ) as file:

        file.write(html)

    print(
        f"Saved {filename}"
    )
```

---

# Expected Output

```text
Downloading classes.html

Saved classes.html

Downloading modules.html

Saved modules.html

Downloading json.html

Saved json.html
```

---

# After Running

Folder:

```text
python_rag/

│
├── raw_docs/
│
│   ├── classes.html
│   ├── modules.html
│   ├── pathlib.html
│   ├── json.html
│   ├── datetime.html
│   ├── os.html
│   ├── functions.html
│   ├── stdtypes.html
│   └── ...
```

---

# What Have We Actually Built?

Not RAG.

Not Retrieval.

Not Embeddings.

---

We built:

```text
Document Ingestion Layer
```

which is the first stage of almost every enterprise AI pipeline.

---

# Interview Question

Why store raw HTML before processing?

Answer:

```text
To separate ingestion from preprocessing,
allowing documents to be cleaned,
reprocessed, and re-embedded without
re-scraping.
```

---

# Interview Question

Why use requests instead of Selenium?

Answer:

```text
Python documentation is static HTML,
so browser automation is unnecessary.
Requests is simpler and faster.
```

# Day 3 — Part 8C

# Understanding The Structure Of Python Documentation

## Before We Write The Cleaner

---

# Why Are We Doing This?

Most beginners write:

```python
from bs4 import BeautifulSoup

soup = BeautifulSoup(html, "html.parser")

text = soup.get_text()
```

and move on.

Looks simple.

But this is actually one of the biggest mistakes in beginner RAG systems.

---

# Let's Think Like An AI Engineer

Suppose we download:

```text
https://docs.python.org/3/library/json.html
```

Question:

What do we actually receive?

Most people imagine:

```text
JSON Documentation

The json module provides...
```

No.

We receive an entire webpage.

---

# What The Website Really Looks Like

Conceptually:

```html
<html>

<head>
    Metadata
</head>

<body>

    Header

    Navigation

    Search Box

    Sidebar

    Main Content

    Footer

</body>

</html>
```

---

# What We Actually Want

Only:

```html
Main Content
```

---

# Let's Visualize It

Imagine this page:

```text
------------------------------------------------

Python Documentation

Search Box

Navigation

Home
Tutorial
Library
Reference

------------------------------------------------

JSON Documentation

The json module provides...

json.dump()

json.dumps()

Examples...

------------------------------------------------

Previous Page

Next Page

Copyright

------------------------------------------------
```

---

# Question

If we embed everything above:

```text
Search Box

Home

Tutorial

Reference

Previous

Next
```

Do these help answer:

```text
What does json.dumps do?
```

No.

---

# This Is Called Noise

Noise = Information that enters embeddings but has no retrieval value.

Example:

```text
Search

Previous

Next

Copyright

Navigation

Breadcrumbs
```

---

# Why Noise Is Dangerous

Imagine 100 pages.

Every page contains:

```text
Previous

Next

Search

Documentation
```

---

What happens?

These words appear thousands of times.

Embedding space starts learning:

```text
Previous

Next

Search
```

as important concepts.

---

Retrieval quality decreases.

---

# What Should A Good Cleaner Do?

Input:

```html
Entire Webpage
```

Output:

```text
Only Educational Content
```

---

# Let's Inspect A Typical Documentation Page

Python docs are built using:

```text
Sphinx
```

---

# What Is Sphinx?

Sphinx is a documentation generator.

Python docs:

```text
Library Docs

Tutorial

Reference

FAQ
```

are generated by Sphinx.

---

# Why Do We Care?

Because Sphinx creates consistent HTML.

Consistent HTML means:

```text
Predictable Structure
```

which makes extraction easier.

---

# Conceptual Structure

A Python docs page roughly looks like:

```html
<body>

    Navigation

    Sidebar

    Main Article

    Footer

</body>
```

---

Inside the article:

```html
<section>

    <h1>JSON Encoder and Decoder</h1>

    <p>...</p>

    <p>...</p>

    <pre>Code Example</pre>

</section>
```

---

# This Is Gold

Why?

Because:

```text
Headings

Paragraphs

Code Examples
```

are exactly what we want to embed.

---

# What We Want To Preserve

---

## Headings

Example:

```text
json.dump
```

Important.

Provides structure.

---

## Paragraphs

Example:

```text
The json module provides...
```

Important.

Contains explanations.

---

## Code Blocks

Example:

```python
json.dumps(data)
```

Important.

Developers often ask questions about code.

---

# What We Want To Remove

---

## Navigation

Example:

```text
Home

Library

Reference
```

Noise.

---

## Search

Example:

```text
Search
```

Noise.

---

## Footer

Example:

```text
Copyright
```

Noise.

---

## Previous / Next Links

Example:

```text
Previous Page

Next Page
```

Noise.

---

# Think About Chunking

Suppose we do:

```python
soup.get_text()
```

on the entire page.

Chunk might become:

```text
Home

Library

Reference

JSON Documentation

The json module...

Previous Page

Next Page
```

---

Question:

Would you want that chunk retrieved?

Not really.

---

Instead:

```text
JSON Documentation

The json module provides...

json.dump()

json.dumps()
```

Perfect.

---

# Therefore The Cleaner's Job Is Not

```text
HTML → Text
```

---

The Real Job Is

```text
HTML → Useful Knowledge
```

---

# Another Important Insight

Many people think:

```text
Chunking determines retrieval quality.
```

Partly true.

But before chunking even begins:

```text
Cleaning determines retrieval quality.
```

---

Bad Cleaning

↓

Bad Chunks

↓

Bad Embeddings

↓

Bad Retrieval

↓

Bad Answers

---

# Let's Design The Cleaner Before Coding

Question:

What should the cleaner output?

---

# Option A

Single giant file.

```text
all_docs.txt
```

Bad.

---

# Why?

Because we lose document identity.

Later:

```text
Which file contained the answer?
```

Impossible to know.

---

# Option B

One cleaned file per source.

Example:

```text
clean_docs/

json.txt

pathlib.txt

modules.txt

classes.txt
```

Much better.

---

# Why?

Metadata becomes easy.

Example:

```python
{
    "source": "json.txt"
}
```

Later:

```text
Answer Source:

json.txt
```

---

# The Data Flow We Want

After cleaning:

```text
raw_docs/

json.html

pathlib.html

classes.html
```

↓

Cleaner

↓

```text
clean_docs/

json.txt

pathlib.txt

classes.txt
```

---

# Before Coding

Let's Verify Our Understanding

Suppose user asks:

```text
What does json.dumps do?
```

---

Will Chroma Search:

```text
json.html
```

directly?

No.

---

Will Chroma Search:

```text
Raw HTML
```

directly?

No.

---

Will Chroma Search:

```text
Cleaned Chunks
```

Yes.

---

Therefore the cleaner is creating the actual knowledge that enters the vector database.

---

# What We'll Build Next

Now we are finally ready to implement:

```text
clean_docs.py
```

It will:

1. Read HTML files from `raw_docs/`
2. Parse them using BeautifulSoup
3. Extract meaningful content
4. Remove noise
5. Save clean text files
6. Preserve source information
7. Prepare documents for chunking

---

# Mental Model To Keep Forever

Most beginners think:

```text
Website
↓
Embeddings
```

Engineers think:

```text
Website
↓
Content Extraction
↓
Knowledge Cleaning
↓
Chunking
↓
Embeddings
↓
Retrieval
```

The better the extraction layer, the better everything downstream becomes.

# Day 3 — Part 8D

# Implementing `clean_docs.py`

## Understanding BeautifulSoup Before Writing Code

---

# STOP — One Important Question

Before writing code, let's answer this:

When we download:

```text
json.html
```

what exactly is inside it?

A file like:

```html
<html>

<head>
...
</head>

<body>

<nav>
Navigation
</nav>

<main>

JSON Documentation

The json module provides...

</main>

<footer>
Copyright
</footer>

</body>

</html>
```

This is called:

```text
HTML Tree
```

---

# HTML Is A Tree

Think of it like folders.

```text
html
│
├── head
│
└── body
     │
     ├── nav
     │
     ├── main
     │
     └── footer
```

---

# Question

How do we navigate this tree?

Example:

```text
Find all paragraphs

Find all headings

Find the article section
```

We need a parser.

---

# Enter BeautifulSoup

BeautifulSoup converts:

```html
<html>
<body>
<h1>Hello</h1>
</body>
</html>
```

into a Python object.

Then we can do:

```python
find()
find_all()
select()
```

to navigate the tree.

---

# Mental Model

Without BeautifulSoup:

```text
HTML
=
Huge String
```

With BeautifulSoup:

```text
HTML
=
Searchable Tree
```

---

# Example

Input:

```html
<h1>JSON</h1>

<p>The json module...</p>
```

BeautifulSoup sees:

```text
Document
│
├── h1
│
└── p
```

and lets us access them.

---

# First BeautifulSoup Example

```python
from bs4 import BeautifulSoup

html = """
<h1>JSON</h1>
<p>The json module provides...</p>
"""

soup = BeautifulSoup(
    html,
    "lxml"
)

print(soup.h1.text)
```

Output:

```text
JSON
```

---

# What Is lxml?

Remember when we installed:

```bash
pip install lxml
```

?

That is an HTML parser.

BeautifulSoup uses it internally.

Think:

```text
BeautifulSoup
=
Interface

lxml
=
Engine
```

---

# Question

Why not use:

```python
soup.get_text()
```

on the entire page?

---

# Example

Suppose page contains:

```html
<nav>

Home

Library

Reference

</nav>

<main>

JSON Documentation

The json module...

</main>

<footer>

Copyright

</footer>
```

---

# get_text() Output

```text
Home

Library

Reference

JSON Documentation

The json module...

Copyright
```

---

Problem:

```text
Navigation
Footer
Noise
```

all become embeddings.

Bad.

---

# Therefore We Need Selective Extraction

We want:

```text
Main Content Only
```

---

# Before Coding

Let's think about Python Docs specifically.

Question:

What is the most valuable information?

---

## Headings

Example:

```text
json.dumps

json.loads

JSONEncoder
```

---

## Paragraphs

Example:

```text
The json module provides...
```

---

## Code Blocks

Example:

```python
json.dumps(data)
```

---

These three things contain nearly all useful knowledge.

---

# Our Extraction Strategy

Instead of:

```python
get_text()
```

we will extract:

```text
Headings

Paragraphs

Code Blocks
```

and ignore everything else.

---

# Cleaner Workflow

Input:

```text
raw_docs/json.html
```

↓

Parse HTML

↓

Find useful elements

↓

Extract text

↓

Save

↓

```text
clean_docs/json.txt
```

---

# What Should The Output Look Like?

Suppose original page contains:

```html
<h1>JSON Encoder and Decoder</h1>

<p>
The json module provides...
</p>

<pre>
json.dumps(data)
</pre>
```

---

Output:

```text
JSON Encoder and Decoder

The json module provides...

json.dumps(data)
```

Beautiful.

Clean.

Semantic.

---

# Let's Build The Cleaner

---

# Step 1

Imports

```python
from pathlib import Path

from bs4 import BeautifulSoup
```

---

# Why pathlib?

We'll be dealing with folders.

Example:

```text
raw_docs/

json.html

pathlib.html
```

Pathlib is safer than string paths.

---

# Step 2

Create Directories

```python
RAW_DIR = Path(
    "raw_docs"
)

CLEAN_DIR = Path(
    "clean_docs"
)

CLEAN_DIR.mkdir(
    exist_ok=True
)
```

---

# What Happens Here?

Suppose:

```text
clean_docs/
```

doesn't exist.

Python creates it.

---

# Step 3

Find All HTML Files

```python
html_files = list(
    RAW_DIR.glob("*.html")
)
```

---

# Example

Suppose folder contains:

```text
json.html

pathlib.html

classes.html
```

Result:

```python
[
    Path("json.html"),
    Path("pathlib.html"),
    Path("classes.html")
]
```

---

# Why glob?

Think:

```text
Search Folder

Return Matching Files
```

---

# Step 4

Loop Through Files

```python
for file_path in html_files:

    print(file_path.name)
```

Output:

```text
json.html

pathlib.html

classes.html
```

---

# Step 5

Read HTML

```python
html = file_path.read_text(
    encoding="utf-8"
)
```

---

Now:

```python
html
```

contains the entire webpage.

---

# Step 6

Parse HTML

```python
soup = BeautifulSoup(
    html,
    "lxml"
)
```

---

Visual:

```text
HTML String

↓

BeautifulSoup

↓

HTML Tree
```

---

# Step 7

Find Useful Tags

Remember our goal.

We want:

```text
Headings

Paragraphs

Code Blocks
```

---

Let's extract them.

```python
elements = soup.find_all(
    [
        "h1",
        "h2",
        "h3",
        "h4",
        "p",
        "pre"
    ]
)
```

---

# What Happens?

BeautifulSoup searches entire page.

Returns:

```python
[
   h1,
   p,
   p,
   h2,
   pre,
   p
]
```

objects.

---

# Step 8

Convert To Text

```python
clean_text = []
```

---

Loop:

```python
for element in elements:

    text = element.get_text(
        strip=True
    )

    if text:

        clean_text.append(
            text
        )
```

---

# Why strip=True?

Removes:

```text
Extra Spaces

Extra Newlines
```

---

# Example

Input:

```text
"\n\nJSON\n\n"
```

Output:

```text
"JSON"
```

---

# Step 9

Join Everything

```python
final_text = "\n\n".join(
    clean_text
)
```

---

Result:

```text
Heading

Paragraph

Code Block

Paragraph
```

nicely separated.

---

# Step 10

Determine Output Filename

Input:

```text
json.html
```

Output:

```text
json.txt
```

---

Code:

```python
output_file = (
    CLEAN_DIR /
    f"{file_path.stem}.txt"
)
```

---

# What Is stem?

```python
json.html
```

↓

```python
json
```

---

# Save Text

```python
output_file.write_text(
    final_text,
    encoding="utf-8"
)
```

---

# Full Cleaner

```python
from pathlib import Path

from bs4 import BeautifulSoup

RAW_DIR = Path(
    "raw_docs"
)

CLEAN_DIR = Path(
    "clean_docs"
)

CLEAN_DIR.mkdir(
    exist_ok=True
)

html_files = list(
    RAW_DIR.glob("*.html")
)

for file_path in html_files:

    print(
        f"Cleaning: {file_path.name}"
    )

    html = file_path.read_text(
        encoding="utf-8"
    )

    soup = BeautifulSoup(
        html,
        "lxml"
    )

    elements = soup.find_all(
        [
            "h1",
            "h2",
            "h3",
            "h4",
            "p",
            "pre"
        ]
    )

    clean_text = []

    for element in elements:

        text = element.get_text(
            strip=True
        )

        if text:

            clean_text.append(
                text
            )

    final_text = "\n\n".join(
        clean_text
    )

    output_file = (
        CLEAN_DIR /
        f"{file_path.stem}.txt"
    )

    output_file.write_text(
        final_text,
        encoding="utf-8"
    )

    print(
        f"Saved: {output_file.name}"
    )
```

---

# Wait — Is This Production Quality?

Not yet.

This is Version 1.

Question:

```text
Will it work?
```

Yes.

Absolutely.

---

Question:

```text
Will it extract only the article?
```

Not necessarily.

Because:

```text
h1
p
pre
```

may exist in navigation areas too.

---

# What A Real Engineer Does Next

Before moving to chunking:

Open:

```text
clean_docs/json.txt
```

and inspect.

Ask:

```text
Did we get useful content?

Did we get junk?

Are headings preserved?

Are code blocks preserved?
```

---

# This Is Extremely Important

Many beginners immediately jump to:

```text
Chunking

Embeddings

Chroma
```

without inspecting outputs.

Huge mistake.

---

# Engineering Rule

After every stage:

```text
Inspect Data
```

Before proceeding.

Always.

---

# What I Want You To Do Next

Run:

```text
scrape_docs.py
```

Then:

```text
clean_docs.py
```

Then open:

```text
clean_docs/json.txt

clean_docs/pathlib.txt

clean_docs/classes.txt
```

and inspect them manually.

# Day 3 — Part 8F

# Building `build_db.py`

## Creating Our First Real Knowledge Base

---

# Before Writing Code

Let's stop and think.

Currently we have:

```text
clean_docs/

├── classes.txt
├── modules.txt
├── pathlib.txt
├── json.txt
├── datetime.txt
├── os.txt
```

Question:

Can an LLM answer questions from these files?

Technically:

```text
Yes
```

But only if we manually copy the files into the prompt.

Not practical.

---

# What We Need

We need a system that can answer:

```text
What is pathlib?

What does json.dumps do?

How do Python modules work?
```

without reading every document every time.

---

# Why Not Search All Documents?

Suppose later we have:

```text
100 documents

5000 chunks
```

Would we send all chunks to the LLM?

No.

---

Reasons:

```text
Too expensive

Too slow

Too many tokens

Context window limitations
```

---

# Therefore We Need

```text
Fast Retrieval
```

---

# The Birth Of Embeddings

Imagine two chunks:

```text
Chunk A

The json module provides...
```

```text
Chunk B

Path objects represent filesystem paths.
```

---

User asks:

```text
What does json.dumps do?
```

Question:

Which chunk is more relevant?

Obviously:

```text
Chunk A
```

---

How can a computer know that?

Answer:

```text
Embeddings
```

---

# Embeddings Refresher

Remember Day 1.

Text:

```text
json.dumps converts Python objects...
```

becomes:

```text
[0.28, -0.41, 0.77, ...]
```

---

This vector captures meaning.

---

# Why This Is Powerful

Question:

```text
What does json.dumps do?
```

becomes:

```text
[0.31, -0.38, 0.81, ...]
```

Notice:

```text
Question Vector
```

and

```text
Relevant Chunk Vector
```

become close together.

---

# Retrieval Becomes Geometry

Instead of:

```text
Searching Words
```

we do:

```text
Searching Vector Space
```

---

# What build_db.py Must Do

Input:

```text
clean_docs/
```

Output:

```text
chroma_db/
```

---

Workflow:

```text
Load Documents

↓

Chunk Documents

↓

Generate Embeddings

↓

Store In Chroma

↓

Persist Database
```

---

# Step 1

Imports

```python
from pathlib import Path

from langchain_community.document_loaders import (
    DirectoryLoader,
    TextLoader
)

from langchain_text_splitters import (
    RecursiveCharacterTextSplitter
)

from langchain_community.embeddings import (
    HuggingFaceEmbeddings
)

from langchain_community.vectorstores import (
    Chroma
)
```

---

# Why DirectoryLoader?

Previously:

```python
TextLoader("json.txt")
```

loaded one file.

Now:

```python
DirectoryLoader(...)
```

loads an entire folder.

---

# Step 2

Folder Paths

```python
DATA_DIR = "clean_docs"

DB_DIR = "chroma_db"
```

---

# Why Separate Them?

```text
clean_docs/
```

contains:

```text
Knowledge
```

---

```text
chroma_db/
```

contains:

```text
Vectors
```

---

Never mix them.

---

# Step 3

Load Documents

```python
loader = DirectoryLoader(
    DATA_DIR,
    glob="*.txt",
    loader_cls=TextLoader
)

documents = loader.load()
```

---

# What Happens?

Suppose:

```text
clean_docs/

json.txt
pathlib.txt
classes.txt
```

---

Output:

```python
[
 Document(...),
 Document(...),
 Document(...)
]
```

---

# Inspect Documents

Always inspect.

```python
print(
    f"Documents Loaded: {len(documents)}"
)
```

---

Expected:

```text
Documents Loaded: 10
```

or similar.

---

# Engineering Rule

Never trust data.

Inspect it.

---

# Step 4

Create Text Splitter

```python
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
```

---

# Why RecursiveCharacterTextSplitter?

This is LangChain's most common splitter.

It tries to split intelligently.

Priority:

```text
Paragraphs

↓

Sentences

↓

Words

↓

Characters
```

---

instead of blindly cutting text.

---

# Create Chunks

```python
chunks = splitter.split_documents(
    documents
)
```

---

# What Happens?

Before:

```text
10 Documents
```

After:

```text
500+ Chunks
```

(depending on size)

---

# IMPORTANT DEBUGGING STEP

Print statistics.

```python
print(
    f"Chunks Created: {len(chunks)}"
)
```

---

# Inspect Chunk

```python
print(
    chunks[0].page_content[:500]
)
```

---

Question:

Why inspect chunks?

Because retrieval quality depends heavily on chunk quality.

---

# Bad Chunk Example

```text
Search

Next

Previous

Home
```

If you see this:

```text
Cleaning Failed
```

---

# Good Chunk Example

```text
The json module provides methods
for serializing and deserializing
Python objects.
```

Perfect.

---

# Step 5

Embedding Model

```python
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)
```

---

# Why This Model?

For learning:

```text
Fast

Small

Free

Reliable
```

---

# Internal Flow

Chunk:

```text
json.dumps converts...
```

↓

Embedding Model

↓

Vector

```text
[0.31, -0.22, ...]
```

---

# Step 6

Create Chroma Database

```python
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=DB_DIR
)
```

---

# Think About This Carefully

For every chunk:

```text
Chunk

↓

Embedding

↓

Vector

↓

Store In Chroma
```

---

Chroma stores:

```text
Chunk Text

+

Vector

+

Metadata
```

---

# Metadata Example

One chunk might contain:

```python
{
    "source":
    "clean_docs/json.txt"
}
```

---

This is incredibly valuable later.

---

# Step 7

Persist Database

Depending on Chroma version:

```python
vector_store.persist()
```

may be optional.

But calling it is fine.

---

# Why Persist?

Without persistence:

Database disappears when program exits.

---

With persistence:

```text
Build Once

Use Forever
```

---

# Final Statistics

Print:

```python
print(
    f"Documents: {len(documents)}"
)

print(
    f"Chunks: {len(chunks)}"
)

print(
    "Database Created Successfully"
)
```

---

# Complete build_db.py

```python
from langchain_community.document_loaders import (
    DirectoryLoader,
    TextLoader
)

from langchain_text_splitters import (
    RecursiveCharacterTextSplitter
)

from langchain_community.embeddings import (
    HuggingFaceEmbeddings
)

from langchain_community.vectorstores import (
    Chroma
)

DATA_DIR = "clean_docs"

DB_DIR = "chroma_db"


print("Loading documents...")

loader = DirectoryLoader(
    DATA_DIR,
    glob="*.txt",
    loader_cls=TextLoader
)

documents = loader.load()

print(
    f"Documents Loaded: {len(documents)}"
)


print("Creating chunks...")

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = splitter.split_documents(
    documents
)

print(
    f"Chunks Created: {len(chunks)}"
)


print(
    "\nSample Chunk:\n"
)

print(
    chunks[0].page_content[:500]
)


print(
    "\nLoading Embedding Model..."
)

embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

print(
    "Creating Chroma Database..."
)

vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=DB_DIR
)

vector_store.persist()

print(
    "\nDatabase Created Successfully!"
)
```

---

# What Happens When You Run It?

Command:

```powershell
python build_db.py
```

---

Expected Output:

```text
Loading documents...

Documents Loaded: 10

Creating chunks...

Chunks Created: 742

Sample Chunk:

The json module provides...

Loading Embedding Model...

Creating Chroma Database...

Database Created Successfully!
```

---

# STOP HERE

Do not build the chatbot yet.

Real engineers verify retrieval first.

---

# Next Step

Before touching an LLM, we build:

```text
debug_retrieval.py
```

Purpose:

```text
Question

↓

Retrieve Chunks

↓

Display Scores

↓

Display Sources

↓

NO LLM
```

This is where you'll finally see whether your embeddings and chunking strategy actually work.

Most tutorials skip this step.

Professional RAG engineers spend a lot of time here.

# Day 3 — Part 8G

# Building `debug_retrieval.py`

## Testing Retrieval Before Using Any LLM

---

# Most RAG Tutorials Make This Mistake

They do:

```text
Documents

↓

Embeddings

↓

Chroma

↓

LLM
```

immediately.

---

This is dangerous.

Because when answers are bad:

```text
Was retrieval bad?

or

Was the LLM bad?
```

You don't know.

---

# Professional Workflow

Always isolate components.

First verify:

```text
Retriever
```

Then verify:

```text
LLM
```

---

# Why?

Suppose user asks:

```text
What does json.dumps do?
```

---

Retriever returns:

```text
Chunk about pathlib
```

---

Then LLM answers incorrectly.

Question:

```text
Who is responsible?
```

Answer:

```text
Retriever
```

---

The LLM never saw the correct information.

---

# Therefore

Before touching Groq:

```text
Question

↓

Embedding

↓

Chroma Search

↓

Retrieved Chunks

↓

STOP
```

No LLM.

---

# What debug_retrieval.py Must Do

Input:

```text
Question
```

Example:

```text
What does json.dumps do?
```

---

Output:

```text
Top 3 Chunks

Similarity Scores

Sources
```

---

# Mental Model

Current State:

```text
Question
```

↓

Embedding

↓

Vector

↓

Chroma

↓

Relevant Chunks

---

No reasoning.

No generation.

No LLM.

---

# Why This Is So Valuable

Suppose user asks:

```text
What is pathlib?
```

---

If retrieval returns:

```text
Path objects represent filesystem paths...
```

Great.

---

If retrieval returns:

```text
Python modules allow code reuse...
```

Problem.

---

We discover the problem immediately.

---

# Architecture

```text
Question

↓

Embedding

↓

Vector Search

↓

Top K Chunks

↓

Print Results
```

---

# Step 1

Imports

```python
from langchain_community.vectorstores import (
    Chroma
)

from langchain_community.embeddings import (
    HuggingFaceEmbeddings
)
```

---

# Why Same Embedding Model?

Remember.

During database creation:

```python
all-MiniLM-L6-v2
```

generated document vectors.

---

Question vectors MUST use:

```python
all-MiniLM-L6-v2
```

as well.

---

Otherwise:

```text
Different Vector Spaces
```

and retrieval breaks.

---

# Step 2

Load Embedding Model

```python
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)
```

---

# Step 3

Load Existing Chroma

```python
db = Chroma(
    persist_directory="chroma_db",
    embedding_function=embeddings
)
```

---

# Think Carefully

We're NOT creating a new database.

We're loading:

```text
Existing Database
```

from disk.

---

# Step 4

Simple Test Query

```python
query = "What does json.dumps do?"
```

---

# Step 5

Similarity Search

```python
results = db.similarity_search_with_score(
    query,
    k=3
)
```

---

# What Happens Internally?

```text
Question

↓

Embedding

↓

Vector

↓

Compare Against All Chunks

↓

Return Top 3
```

---

# What Is Returned?

Something like:

```python
[
    (Document(...), score),
    (Document(...), score),
    (Document(...), score)
]
```

---

# Step 6

Print Results

```python
for i, (doc, score) in enumerate(results):

    print("=" * 80)

    print(
        f"Result {i+1}"
    )

    print(
        f"Score: {score}"
    )

    print(
        f"Source: "
        f"{doc.metadata.get('source')}"
    )

    print()

    print(
        doc.page_content[:1000]
    )
```

---

# Why Print Metadata?

Remember:

```python
{
    "source":
    "clean_docs/json.txt"
}
```

---

This tells us:

```text
Where answer came from
```

---

# Full Version

```python
from langchain_community.vectorstores import (
    Chroma
)

from langchain_community.embeddings import (
    HuggingFaceEmbeddings
)

embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

db = Chroma(
    persist_directory="chroma_db",
    embedding_function=embeddings
)

query = "What does json.dumps do?"

results = db.similarity_search_with_score(
    query,
    k=3
)

for i, (doc, score) in enumerate(results):

    print("\n")

    print("=" * 80)

    print(
        f"Result {i+1}"
    )

    print(
        f"Score: {score}"
    )

    print(
        f"Source: "
        f"{doc.metadata.get('source')}"
    )

    print()

    print(
        doc.page_content[:1000]
    )
```

---

# Run It

```powershell
python debug_retrieval.py
```

---

# What Good Retrieval Looks Like

Query:

```text
What does json.dumps do?
```

---

Result 1:

```text
Source: json.txt

json.dumps serializes an object
to a JSON formatted string...
```

Excellent.

---

# What Bad Retrieval Looks Like

Query:

```text
What does json.dumps do?
```

---

Result 1:

```text
Source: pathlib.txt

Path objects represent...
```

Problem.

---

# Testing Strategy

After running, try:

---

## Test 1

```text
What does json.dumps do?
```

Expected:

```text
json.txt
```

---

## Test 2

```text
What is pathlib?
```

Expected:

```text
pathlib.txt
```

---

## Test 3

```text
How do Python classes work?
```

Expected:

```text
classes.txt
```

---

## Test 4

```text
What are modules?
```

Expected:

```text
modules.txt
```

---

# Important Observation

At this stage:

```text
You have built a search engine.
```

Not a chatbot.

Not an AI assistant.

Not an agent.

---

Just:

```text
Semantic Search
```

---

# This Is Actually The Heart Of RAG

Many people think:

```text
LLM = RAG
```

Wrong.

---

RAG is really:

```text
Retrieval

+

Generation
```

And retrieval usually determines most of the quality.

---

# Interview Question

Why test retrieval before connecting an LLM?

Answer:

```text
To isolate retrieval quality from generation quality.

If retrieval is poor,
the LLM never receives the correct context.
```

---

# End Of Part 8G

Do NOT proceed to Groq yet.

Run:

```powershell
python debug_retrieval.py
```

Test at least 5 different queries.

Paste the results.

We will inspect:

```text
Chunk Quality

Source Quality

Retrieval Quality

Metadata Quality
```